# Sorting and Renaming Court Documents

A downloaded litigation library usually mixes complaints with other filings, under inconsistent names. This notebook sorts a folder containing:

| Item | Example |
|---|---|
| Single PDFs | `Moss v. City Of Riverton, N.J. - Entry null.pdf` |
| Folders of attachments | `Doe v. County Sheriff et al - Entry 1/` (complaint, exhibits, civil cover sheet) |
| ZIP archives | `Roe v. Smith et al - Entry 1.zip` |
| Later docket entries | `... - Entry 18`, `... - Entry 41` (amended complaints, orders, motions) |
| Duplicates and same-named files | `... - Entry 1-2`; several different files all named `1.pdf` |

**Steps:**
1. Extract ZIP archives into a staging folder.
2. Find every PDF and record its case from the folder or file name.
3. **Read each PDF's content** and classify it from its own title, typical complaint features, and distinctive phrases, skipping provider cover pages and docket reports. File names are never used to decide what a document is.
4. Read the case number from the document (court header or provider cover page).
5. Detect duplicates by content: identical files, or the same text saved twice within a case.
6. Choose one complaint per case (by default the operative, most recently amended complaint).
7. Copy the files into sorted folders under standard names, `{case_number}_{short_case_name}_{document_type}_{descriptor}.pdf`, and write a manifest explaining every decision.
8. **Apply your corrections:** edit the manifest's override columns and run again.

**Order of work:** Step 1 (dry run) → Step O1 (check for scans) → Step O2 (OCR, if needed) → Step O3 (dry run again) → Step 2 (review and correct; re-run Step 1 after corrections) → Step 3 (copy) → Step 4 (final check of cases without a complaint; then Steps 1 and 3 again) → Step 5 (validation summary) → Step 6 (validation test set for hand coding).

**Do not use Runtime → Run all**: Step 3 copies files and should run only after the review. Use *Runtime → Run before* on the Step 1 cell, then run the steps one at a time.

**The original folder is never modified.**

---
### Status and what to run next

**The sorting is complete** (all steps run, corrections applied): 664 PDFs → 484 cases → **466 complaints** in `COMPLAINTS_SORTED/complaints/`. You do **not** need to re-run Steps 1–5 unless you change the manifest.

**Next, in this notebook:** run the **Setup** and **Settings** cells, then **Step 3b** (screen the complaints for wrong documents, e.g. a magistrate's recommendation selected as a complaint), review `flagged_documents.csv`, run **Step 3c**, and re-run **Steps 1 and 3**. Step 6b (the gold set) is done.

**Then, in the other notebooks:**
1. `02_Complaint_RAG_Extraction_validation.ipynb`: extract the gold complaints (GPU runtime; writes `validation_complaints/library_analysis.xlsx`).
2. `03_Validation.ipynb`: compare the extraction with the hand coding (Steps 1, 3–7; its Step 2 is replaced by Step 6b here).
3. Later: the same extraction notebook on all 466 complaints (`COMPLAINT_FOLDER = .../COMPLAINTS_SORTED/complaints`).

## Setup

In [ ]:
!pip install -q pymupdf openai

In [ ]:
import hashlib
import os
import re
import shutil
import zipfile
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
from tqdm.auto import tqdm   # progress bars (pre-installed in Colab)

from google.colab import drive
drive.mount('/content/drive')

## Settings

- `SOURCE_DIR`: the folder holding all downloads, here `COMPLAINTS_ALL` with its four batch folders (`FEB_3-4_CASES`, `FEBRUARY_5_CASES`, `FEBRUARY_6_CASES`, `FEBRUARY_7_CASES`). All subfolders are searched, and each document's batch folder is recorded in the manifest.
- `OUTPUT_DIR`: where the sorted copies and the manifest are written. It must be **outside** `SOURCE_DIR`; otherwise the sorted copies would be read again as input on the next run.
- `PRIMARY_RULE`: `"latest"` chooses the operative (most recently amended) complaint for each case, usually the most complete account of the incidents; `"original"` chooses the first complaint filed.
- `MAX_PAGES_TO_SCAN`: provider PDFs can contain a long docket report before the document begins.
- `USE_LLM_FOR_CASE_NAMES`: optional. Case names are read from the legal-database citation line and the complaint's caption; for captions the rules cannot parse, an OpenAI model (`LLM_MODEL`) can read the parties, and its answer is accepted only if both names appear in the page text. Needs an API key in `config.json` or the `OPENAI_API_KEY` environment variable.

In [ ]:
SOURCE_DIR = Path("/content/drive/MyDrive/COMPLAINTS_ALL")      # contains the four batch folders
OUTPUT_DIR = Path("/content/drive/MyDrive/COMPLAINTS_SORTED")   # created if missing; must be OUTSIDE SOURCE_DIR
PRIMARY_RULE = "latest"      # "latest" (operative complaint) or "original"
MAX_PAGES_TO_SCAN = 60

# Optional: an OpenAI model reads case names from captions the rules cannot parse (each answer is accepted only if
# both party names appear in the page text). About 500 short requests cost well under $1 with gpt-4o-mini.
USE_LLM_FOR_CASE_NAMES = False
LLM_MODEL = "gpt-4o-mini"
OPENAI_CONFIG = Path("/content/config.json")        # {"OPENAI_API_KEY": "sk-..."}; or set OPENAI_API_KEY

print("Source folder found:", SOURCE_DIR.exists())
print("Batch folders:", sorted(p.name for p in SOURCE_DIR.iterdir() if p.is_dir()) if SOURCE_DIR.exists() else "-")

### What is in each batch folder

A quick inventory before processing: PDFs, ZIP archives and case folders per batch. Large batches take longer, since every PDF is opened and read from Google Drive.

In [ ]:
inventory = []
for batch in sorted(p for p in SOURCE_DIR.iterdir() if p.is_dir()):
    files = [f for f in batch.rglob("*") if f.is_file()]
    inventory.append({
        "batch": batch.name,
        "PDFs": sum(f.suffix.lower() == ".pdf" for f in files),
        "ZIP archives": sum(f.suffix.lower() == ".zip" for f in files),
        "other files": sum(f.suffix.lower() not in (".pdf", ".zip") for f in files),
        "case folders": sum(1 for d in batch.iterdir() if d.is_dir()),
    })
inventory = pd.DataFrame(inventory)
display(inventory)
print("Total PDFs:", inventory["PDFs"].sum(), "| ZIP archives:", inventory["ZIP archives"].sum())

## Classification Rules

Each PDF is classified from its content, in order of strength of evidence:

1. **The document's own title**, in title-like lines near the top (capitalized or standalone): "COMPLAINT", "FIRST AMENDED COMPLAINT", "VERIFIED COMPLAINT", "PETITION" for complaints; for other documents, titles such as "CIVIL COVER SHEET", "EXHIBIT", "SUMMONS", "MOTION", "OPPOSITION", "REPLY", "ORDER", "OPINION", "DECLARATION", "DEPOSITION", "SUBPOENA", "SETTLEMENT AGREEMENT", "CONSENT DECREE". A sentence that merely mentions "the complaint" is not a title.
2. **Complaint body signals**: numbered paragraphs, a parties section, jurisdiction, causes of action or 42 U.S.C. § 1983, a prayer for relief, a jury demand.
3. **Distinctive body phrases**, used only when no title is found: "it is hereby ordered" (order), "I declare under penalty of perjury" (declaration), "subscribed and sworn" (affidavit), "you are commanded" (subpoena), "respectfully moves" (motion), and others.

Provider title pages and docket reports (e.g., a Bloomberg Law title page, "Docket Text") are skipped, so the document's first page is found even when the complaint starts on page 2 or later. If the first page carries no document title at all (e.g., an unrecognized title sheet listing the parties), the next two pages are checked for a complaint title. Titles may carry modifiers ("VERIFIED COMPLAINT", "PLAINTIFF'S ORIGINAL COMPLAINT", "CIVIL RIGHTS COMPLAINT", "FIRST AMENDED COMPLAINT"), run up to 20 words ("…COMPLAINT FOR DAMAGES AND DEMAND FOR JURY TRIAL"), or be split over two lines ("FIRST AMENDED" / "COMPLAINT").

The case number is read from the court's header (`1:22-cv-04065` becomes `2022cv04065`) or a provider cover page; common state formats (e.g., `24STCV33992`, New Jersey's `BER-L-001234-24`) are also recognized. State captions often list many defendants in a two-column layout, which pushes the title (e.g., "COMPLAINT WITH JURY DEMAND") far down the first page; the whole first page is searched, and a page with a court caption ("Plaintiff, vs. … Defendants.") is never mistaken for a provider title page, even when it says "Docket No.".

In [ ]:
NAME_RE = re.compile(r"^(?P<case>.+?)\s+-\s+Entry\s+(?P<entry>\d+|null)(?P<copy>-\d+)?$", re.I)

# Provider cover pages and docket reports: skipped when looking for the document's first page
DOCKET_PAGE_RE = re.compile(r"bloomberg law|docket report|docket text|date filed\s*#|case details|"
                            r"proceedings? (include|for case)|assigned to:|demand:\s*\$", re.I)

# Provider cover/title pages (e.g. a Bloomberg Law title page before the complaint): skipped as well
COVER_PAGE_RE = re.compile(r"bloomberg|westlaw|lexis|courtlink|court docket|general information|"
                           r"docket no\.[^\n]*\([^\n]*\d{4}\)|"           # provider citation: "Docket No. ... (D.N.J. Feb 6, 2024)"
                           r"nature of suit|jurisdictional basis|retrieved|downloaded|printed by|pulled by", re.I)
MODIFIER_RE = re.compile(r"\b(first|second|third|fourth|fifth|\d+(st|nd|rd|th)|amended|verified|original|"
                         r"civil rights|class action|plaintiff'?s'?)\b", re.I)

# Document titles, checked on short lines near the top of the first content pages
COMPLAINT_TITLE_RE = re.compile(
    r"\b(?:(?P<ord>first|second|third|fourth|fifth|\d+(?:st|nd|rd|th))\s+)?"
    r"(?P<amended>amended\s+)?(?:verified\s+)?(?:class\s+action\s+)?(?:civil\s+rights\s+)?"
    r"complaint\b(?!\s+(?:form|department|division|unit|no\b|number|investigat))", re.I)
PETITION_TITLE_RE = re.compile(r"\b(?:verified\s+)?petition\b(?!\s+for\s+(?:removal|writ))", re.I)
OTHER_TITLE_RE = re.compile(
    r"civil cover sheet|\bjs[- ]?44\b|summons in a civil action|^\s*summons\b|^\s*exhibit\b|"
    r"notice of removal|\bmotion\b|\border\b|memorandum|declaration of|affidavit|\banswer\b|stipulation|"
    r"certificate of service|in forma pauperis|\bjudgment\b|\btranscript\b|report and recommendation|"
    r"notice of (appearance|filing|voluntary dismissal)|\bproposed\b|\bbrief\b|\bopposition\b|^\s*reply\b|"
    r"\bdeposition\b|\bsubpoena\b|settlement agreement|consent decree|\bopinion\b", re.I)

# Categories for documents that are not complaints (first match wins; checked on title lines)
CATEGORY_PATTERNS = [
    ("civil cover sheet", r"civil cover sheet|\bjs[- ]?44\b"),
    ("consent decree", r"consent decree"),
    ("settlement agreement", r"settlement agreement"),
    ("subpoena", r"\bsubpoena\b"),
    ("deposition", r"\bdeposition\b"),
    ("opinion", r"\bopinion\b"),
    ("opposition", r"\bopposition\b|in opposition to"),
    ("reply", r"^\s*reply\b|reply (brief|memorandum|in support)"),
    ("summons", r"summons"),
    ("exhibit", r"^\s*exhibit\b|\bexhibit\s+[a-z0-9]{1,3}\b"),
    ("notice of removal", r"notice of removal"),
    ("motion", r"\bmotion\b"),
    ("memorandum or brief", r"memorandum|\bbrief\b"),
    ("order or judgment", r"\border\b|\bjudgment\b|report and recommendation"),
    ("declaration or affidavit", r"declaration of|affidavit"),
    ("answer", r"\banswer\b"),
    ("in forma pauperis application", r"in forma pauperis"),
    ("notice", r"^\s*notice of"),
    ("stipulation", r"stipulation"),
    ("certificate of service", r"certificate of service"),
    ("transcript", r"transcript"),
]


# Distinctive body phrases: secondary evidence for documents without a recognizable title
BODY_PHRASES = {
    "order or judgment": r"it is (hereby )?ordered|the court (hereby )?orders|so ordered|judgment is (hereby )?entered",
    "declaration or affidavit": r"declare under penalty of perjury|subscribed and sworn|being duly sworn|\baffiant\b",
    "subpoena": r"you are (hereby )?commanded",
    "deposition": r"examination by|\bdeponent\b|the witness:",
    "settlement agreement": r"settlement agreement|release of (all )?claims",
    "transcript": r"the court:|reporter'?s transcript|court reporter",
    "motion": r"respectfully moves|moves this court|hereby moves",
}


def document_category(title):
    for name, rx in CATEGORY_PATTERNS:
        if re.search(rx, title or "", re.I):
            return name
    return "other"


# Body signals typical of a complaint
BODY_SIGNALS = {
    "numbered paragraphs": re.compile(r"(?m)^\s*\d{1,3}\.\s+\S"),
    "parties section": re.compile(r"\bparties\b", re.I),
    "jurisdiction section": re.compile(r"jurisdiction and venue|\bjurisdiction\b", re.I),
    "prayer for relief": re.compile(r"prayer for relief|wherefore|relief requested", re.I),
    "jury demand": re.compile(r"jury (trial )?demand|demand(s)? (for )?(a )?trial by jury", re.I),
    "causes of action": re.compile(r"cause of action|\bcount\s+[ivx\d]+\b|42 u\.?s\.?c\.?\s*§?\s*1983", re.I),
}

# Case numbers: federal (1:22-cv-04065) and common state formats (24STCV33992)
FED_CASE_RE = re.compile(r"\b\d{1,2}:(\d{2})-([a-z]{2,4})-(\d{3,6})\b", re.I)
NY_INDEX_RE = re.compile(r"index\s+(?:no\.?|number)\s*:?\s*(\d{3,7})\s*/\s*(\d{2,4})", re.I)   # New York state
STATE_CASE_RE = re.compile(r"\b(\d{2}[A-Z]{2,6}\d{4,8}|[A-Z]{3}-[A-Z]{1,2}-\d{3,6}-\d{2})\b")   # e.g. 24STCV33992, BER-L-001234-24

# ECF header: "Document 1" (main document) vs "Document 1-2" (attachment to document 1)
ECF_DOC_RE = re.compile(r"Document\s+(\d+)(?:-(\d+))?\s+Filed", re.I)

ORDINALS = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5}

## Finding Files: ZIP Extraction, Case Names and Duplicates

In [ ]:
def extract_zips(source, staging):
    """Extract every ZIP archive into staging/<archive name>/ (the original archives are left untouched)."""
    zips = sorted(source.rglob("*.zip"))
    for z in tqdm(zips, desc="Extracting ZIP archives", unit="zip", disable=not zips):
        target = staging / z.relative_to(source).parent / z.stem     # mirrors the batch folders, avoids name clashes
        if not target.exists():
            target.mkdir(parents=True)
            try:
                with zipfile.ZipFile(z) as archive:
                    archive.extractall(target)
            except zipfile.BadZipFile:
                tqdm.write(f"⚠ Could not open archive: {z.relative_to(source)}")


def case_and_entry(path, root):
    """Case name and docket entry from the nearest file or folder name of the form '<case> - Entry <n>'."""
    candidates = [path.stem] + [p.name for p in path.relative_to(root).parents if p.name]
    for name in candidates:
        m = NAME_RE.match(name.strip())
        if m:
            entry = None if m.group("entry").lower() == "null" else int(m.group("entry"))
            return m.group("case").strip(), entry, bool(m.group("copy"))
    return path.stem, None, False


def text_fingerprint(path, max_pages=10):
    """Hash of the normalized text of the first pages: identifies the same document saved twice,
    even when the files differ byte by byte (e.g. re-downloads with new metadata)."""
    try:
        with fitz.open(path) as doc:
            text = " ".join(doc[i].get_text() for i in range(min(max_pages, len(doc))))
    except Exception:
        return None
    text = re.sub(r"\s+", " ", text).strip().lower()
    return hashlib.sha256(text.encode()).hexdigest() if len(text) > 200 else None


def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

## Classifying Each PDF

Each document is labeled:
- **complaint**, with *high* confidence when a complaint title is supported by at least two body signals, otherwise *medium*;
- **other**, with a category from its title (high confidence) or from a distinctive body phrase (medium); documents with neither are *unidentified* (low);
- **needs review**, when the evidence conflicts, when the body looks like a complaint but no title is found, or when the PDF has no text layer (a scan that needs OCR).

The amendment level (0 = original, 1 = first amended, 2 = second amended, …) is read from the title.

In [ ]:
def looks_like_cover(text, has_more_pages=True):
    """A provider title page: names the provider or docket, or is a short page with no numbered paragraphs,
    no court header and no document title, followed by more pages (a one-page cover sheet or order is not skipped)."""
    numbered = len(re.findall(r"(?m)^\s*\d{1,3}\.\s+\S", text))
    # a court caption ("Plaintiff, v. ... Defendants.") is the document itself, never a provider page
    if (re.search(r"(?mi)^\s*vs?\.?\s*$", text) and re.search(r"\bplaintiffs?\b", text, re.I)
            and re.search(r"\bdefendants?\b", text, re.I)):
        return False
    if COVER_PAGE_RE.search(text) and numbered < 3:
        return True
    other_titled = any(OTHER_TITLE_RE.search(l) for l in title_lines(text, 40)[:15])
    caption = re.search(r"\bplaintiffs?\b", text, re.I) and re.search(r"\bdefendants?\b", text, re.I)
    # a short page that only names the document (e.g. "Complaint") without a caption is a provider cover
    return (has_more_pages and len(text.split()) < 120 and numbered == 0
            and not ECF_DOC_RE.search(text) and not other_titled and not caption)


def first_content_pages(doc, n=2):
    """Find where the document itself starts, skipping provider title pages and docket reports.
    The start is the first page whose top lines carry a complaint title, or else the first page that is
    not a provider page. Returns the page index and the text of n pages from there."""
    first_content = None
    for i in range(min(len(doc), MAX_PAGES_TO_SCAN)):
        text = doc[i].get_text()
        if not text.strip() or DOCKET_PAGE_RE.search(text):
            continue
        more = i < len(doc) - 1
        if any(is_complaint_title(l) or PETITION_TITLE_RE.search(l) for l in title_lines(text, 40)) \
                and not looks_like_cover(text, more):
            first_content = i
            break
        if looks_like_cover(text, more):
            continue
        if first_content is None:
            first_content = i
            break
    if first_content is None:
        return None, ""
    # A title sheet that is not recognized as a provider page (e.g. it lists the parties): if the first
    # content page carries no document title, look at the next two pages for a complaint title
    first_titles = title_lines(doc[first_content].get_text(), 40)[:15]
    if not any(is_complaint_title(l) or PETITION_TITLE_RE.search(l) or OTHER_TITLE_RE.search(l)
               for l in first_titles):
        for j in range(first_content + 1, min(first_content + 3, len(doc))):
            if any(is_complaint_title(l) or PETITION_TITLE_RE.search(l) for l in title_lines(doc[j].get_text(), 40)):
                first_content = j
                break
    return first_content, "\n".join(doc[j].get_text() for j in range(first_content, min(first_content + n, len(doc))))


def is_complaint_title(line):
    """A complaint title, not another document about the complaint ("MOTION TO DISMISS ... COMPLAINT",
    "ANSWER TO COMPLAINT", "EXHIBIT A - COMPLAINT")."""
    m = COMPLAINT_TITLE_RE.search(line)
    if not m:
        return False
    before = line[:m.start()]
    return not OTHER_TITLE_RE.search(before) and not re.search(r"\b(answer|response|reply|opposition)\b",
                                                                before, re.I)


def is_title_like(line):
    """Document titles are set in capitals or stand alone on a short line (not inside a sentence)."""
    letters = [c for c in line if c.isalpha()]
    mostly_caps = letters and sum(c.isupper() for c in letters) / len(letters) >= 0.7
    # short standalone lines count too, but not the end of a sentence ("...demands judgment.")
    return bool(mostly_caps) or (len(line.split()) <= 5 and not line.rstrip().endswith((".", ",", ";")))


def title_lines(text, max_lines=120, max_words=20):
    """Short, title-like lines near the top of the page: where captions and document titles appear.
    Captions with long party lists (e.g. two-column state captions) can push the title far down the page."""
    lines = [l.strip() for l in text.split("\n") if l.strip()][:max_lines]
    return [l for l in lines if len(l.split()) <= max_words and is_title_like(l)]


def classify_pdf(path):
    """Return a dict with the classification, confidence, amendment level and the reasons."""
    result = {"pages": None, "case_number": None, "content_starts_page": None, "complaint_pages": None,
              "ecf_document": None, "title": None,
              "doc_category": None,
              "amendment_level": None, "doc_type": "needs_review", "confidence": "low", "reasons": ""}
    try:
        doc = fitz.open(path)
    except Exception as e:
        result["reasons"] = f"cannot open PDF ({e})"
        return result

    with doc:
        result["pages"] = len(doc)
        start, text = first_content_pages(doc)
        if start is None:
            result["reasons"] = "no text layer in the first pages (scanned without OCR?)"
            return result
        result["content_starts_page"] = start + 1
        body = "\n".join(doc[j].get_text() for j in range(start, min(start + 6, len(doc))))
        # case number: from the content pages first, then from provider cover/docket pages
        head = text + "\n" + "\n".join(doc[j].get_text() for j in range(min(start, 3)))
        fed = FED_CASE_RE.search(head)
        if fed:
            result["case_number"] = f"20{fed.group(1)}{fed.group(2).lower()}{fed.group(3)}"
        else:
            st = STATE_CASE_RE.search(head)
            ny = NY_INDEX_RE.search(head)
            result["case_number"] = (st.group(1).lower() if st else
                                     f"ny{ny.group(1)}_{ny.group(2)}" if ny else None)

    ecf = ECF_DOC_RE.search(text)
    if ecf:
        result["ecf_document"] = ecf.group(1) + (f"-{ecf.group(2)}" if ecf.group(2) else "")

    titles = title_lines(text)
    # positions are tracked by index, since a split title is joined into a new string
    complaint_idx = next((k for k, l in enumerate(titles) if is_complaint_title(l)), None)
    complaint_title = titles[complaint_idx] if complaint_idx is not None else None
    if complaint_title and complaint_idx > 0:  # a title split over two lines, e.g. "FIRST AMENDED" / "COMPLAINT"
        prev = titles[complaint_idx - 1]
        if MODIFIER_RE.search(prev) and len(prev.split()) <= 4 and not COMPLAINT_TITLE_RE.search(prev):
            complaint_title = prev + " " + complaint_title
    petition_idx = next((k for k, l in enumerate(titles) if PETITION_TITLE_RE.search(l)), None)
    petition_title = titles[petition_idx] if petition_idx is not None else None
    other_idx = next((k for k, l in enumerate(titles[:15]) if OTHER_TITLE_RE.search(l)), None)
    other_title = titles[other_idx] if other_idx is not None else None
    signals = [name for name, rx in BODY_SIGNALS.items() if rx.search(body)]

    reasons = []
    if complaint_title or petition_title:
        title = complaint_title or petition_title
        title_idx = complaint_idx if complaint_title else petition_idx
        result["title"] = title
        m = COMPLAINT_TITLE_RE.search(title)
        if m and m.group("amended"):
            ordinal = (m.group("ord") or "first").lower()
            result["amendment_level"] = ORDINALS.get(ordinal) or int(re.sub(r"\D", "", ordinal) or 1)
        else:
            result["amendment_level"] = 0
        result["doc_category"] = ("amended complaint" if result["amendment_level"] else
                                  "petition" if not complaint_title else "complaint")
        reasons.append(f"title: '{title[:80]}'")
        if signals:
            reasons.append("body: " + ", ".join(signals))
        # an exhibit or cover sheet that merely mentions "complaint" in a short line
        if other_title and other_idx < title_idx - (1 if title != titles[title_idx] else 0):
            result.update(doc_type="needs_review", confidence="low")
            reasons.append(f"also titled '{other_title[:60]}' above the complaint title")
        elif len(signals) >= 2:
            result.update(doc_type="complaint", confidence="high")
        else:
            result.update(doc_type="complaint", confidence="medium")
    elif other_title:
        result.update(doc_type="other", confidence="high", title=other_title,
                      doc_category=document_category(other_title))
        reasons.append(f"title: '{other_title[:80]}'")
    elif len(signals) >= 3:
        result.update(doc_type="needs_review", confidence="low")
        reasons.append("no complaint title, but complaint-like body: " + ", ".join(signals))
    else:
        phrase = next(((cat, m.group(0)) for cat, rx in BODY_PHRASES.items()
                       for m in [re.search(rx, body, re.I)] if m), None)
        if phrase:
            result.update(doc_type="other", confidence="medium", doc_category=phrase[0])
            reasons.append(f"no title found; body phrase '{phrase[1]}'")
        else:
            result.update(doc_type="other", confidence="low", doc_category="unidentified")
            reasons.append("no complaint title, no recognizable document type")

    if result["ecf_document"] and "-" in result["ecf_document"]:
        reasons.append(f"ECF attachment {result['ecf_document']} (e.g. an exhibit, or a state complaint "
                       f"attached to a notice of removal)")
    result["reasons"] = "; ".join(reasons)
    return result

### Complaints Inside Other Documents

Some files contain the complaint inside another document:
- **New York state "Summons and Complaint" filings**: page 1 is the summons, the complaint follows;
- **Notices of removal**: the defendant's notice comes first, with the plaintiff's state complaint attached as an exhibit;
- **Provider bundles**: a docket report, then the complaint, then exhibits and later filings.

When a document is classified as a summons, a notice of removal, or unidentified, the notebook searches its pages (up to 150) for the complaint's own first page: a page with a caption ("Plaintiff … Defendant") and a complaint title near the top. The complaint ends before the next filing starts (a different ECF document number, a provider page such as Bloomberg's "General Information" or "Part / Description" pages between documents, or a page titled exhibit, summons, civil cover sheet, certificate, notice or order). The document is then classified as a complaint (medium confidence), the page range is recorded in `complaint_pages`, and **only those pages are copied** in Step 3, so the extraction pipeline reads the complaint without the surrounding material.

The same page-range detection is applied to documents already classified as complaints: when a file also contains a docket report before the complaint, or exhibits and later filings after it, only the complaint's pages are copied. New York state index numbers (e.g., `Index No. 512345/2023`) are recognized as case numbers.

In [ ]:
# ---- Complaints inside other documents ----
# Some files hold the complaint inside another document: New York state "Summons and Complaint" filings
# (summons on page 1), notices of removal (the state complaint attached as an exhibit), and provider
# bundles (docket report, then the complaint, then exhibits). The complaint's own first page is found by
# its caption ("Plaintiff ... Defendant") together with a complaint title near the top of the page.
INSIDE_SEARCH_CATEGORIES = {"summons", "notice of removal", "unidentified"}
MAX_PAGES_INSIDE = 150


def has_caption(text):
    return bool(re.search(r"\bplaintiffs?\b", text, re.I) and re.search(r"\bdefendants?\b", text, re.I))


def find_complaint_inside(doc, first_page=0, max_pages=MAX_PAGES_INSIDE):
    """First page (0-based) with a complaint title and a caption, the last page of the complaint, and its title.
    The complaint ends before the next page that starts another filing: a different ECF document number,
    or a page titled as an exhibit, summons, civil cover sheet, certificate, notice or order."""
    start = title = None
    for i in range(first_page, min(len(doc), first_page + max_pages)):
        text = doc[i].get_text()
        if not text.strip() or DOCKET_PAGE_RE.search(text):
            continue
        lines = title_lines(text)
        t = next((l for l in lines if is_complaint_title(l)), None)
        if t and has_caption(text):
            k = lines.index(t)
            if k > 0 and MODIFIER_RE.search(lines[k - 1]) and len(lines[k - 1].split()) <= 4 \
                    and not COMPLAINT_TITLE_RE.search(lines[k - 1]):
                t = lines[k - 1] + " " + t
            start, title = i, t
            break
    if start is None:
        return None
    first_ecf = ECF_DOC_RE.search(doc[start].get_text())
    start_is_provider = bool(COVER_PAGE_RE.search(doc[start].get_text()))
    end = len(doc) - 1
    for j in range(start + 1, len(doc)):
        text = doc[j].get_text()
        # a provider page between documents (e.g. Bloomberg "General Information" or "Part / Description")
        if not start_is_provider and (COVER_PAGE_RE.search(text) or DOCKET_PAGE_RE.search(text)
                                      or re.search(r"(?m)^\s*Part\s*\n\s*Description\s*$", text)):
            end = j - 1
            break
        ecf = ECF_DOC_RE.search(text)
        if first_ecf and ecf and ecf.group(0) != first_ecf.group(0):
            end = j - 1
            break
        top = title_lines(text, 25)[:8]
        if any(re.search(r"^\s*exhibit\b|summons|civil cover sheet|certificate of|^\s*notice of|\border\b", l, re.I)
               for l in top):
            end = j - 1
            break
    return start, end, title


def look_inside(path, result):
    """If a document was classified as a summons, a notice of removal, or unidentified, look for a complaint
    inside it and, if one is found, reclassify the document as that complaint, with its page range.
    For documents already classified as complaints, record the complaint's page range when the file also
    holds other material (a docket report before it, exhibits or later filings after it)."""
    if result["doc_type"] == "complaint":
        start0 = (result.get("content_starts_page") or 1) - 1
        with fitz.open(path) as doc:
            n = len(doc)
            found = find_complaint_inside(doc, start0, max_pages=3)
        if found and (found[0] > 0 or found[1] < n - 1):
            result["complaint_pages"] = f"{found[0] + 1}-{found[1] + 1}"
            result["reasons"] += f"; complaint on pages {found[0] + 1}-{found[1] + 1} of {n}, only these are copied"
        return result
    if (result.get("doc_category") or "") not in INSIDE_SEARCH_CATEGORIES:
        return result
    with fitz.open(path) as doc:
        found = find_complaint_inside(doc, 0)
    if not found:
        return result
    start, end, title = found
    m = COMPLAINT_TITLE_RE.search(title)
    level = 0
    if m and m.group("amended"):
        ordinal = (m.group("ord") or "first").lower()
        level = ORDINALS.get(ordinal) or int(re.sub(r"\D", "", ordinal) or 1)
    container = result.get("doc_category")
    result.update(
        doc_type="complaint", confidence="medium", title=title, amendment_level=level,
        doc_category="amended complaint" if level else "complaint",
        complaint_pages=f"{start + 1}-{end + 1}", content_starts_page=start + 1,
        reasons=f"complaint found inside a {container} on pages {start + 1}-{end + 1} "
                f"(title '{title[:60]}'); only these pages are copied; " + result["reasons"])
    return result

## Standard File Names

Copied files are renamed to `{case_number}_{short_case_name}_{document_type}_{descriptor}.pdf`, for example:

| Document | New name |
|---|---|
| Complaint, ECF document 1 | `2023cv00001_bagby_v_clayton_sheriff_complaint_ecf1.pdf` |
| Exhibit attached to it | `2023cv00001_bagby_v_clayton_sheriff_exhibit_ecf1_1.pdf` |
| Second amended complaint | `2020cv00001_burnett_v_michigan_amended_complaint_2.pdf` |

- **Case number** from the document; documents without one (e.g., cover sheets) take the number found elsewhere in the same case, and `nocase` is used only if none is found.
- **Short case name**: first plaintiff's surname and the first distinctive words of the first defendant, taken, in order of reliability, from the **legal-database citation line** in the original download (*Poe v. City of Springfield et al, Docket No. …*), the **complaint's caption** (e.g. *JORDAN AVERY, Plaintiff, v. CITY OF RIVERTON …* → `avery_v_riverton`; two-column captions are also read from the left column alone), or optionally an **LLM reading of the caption** accepted only if both names appear in the page text; download file names (often truncated or generic: `14.pdf`, `1 copy 13`) are used only as a last resort. Names in capitals are converted to normal capitalization, keeping acronyms (NYPD) and names like LaRue or McDonald. The full name and its source are kept in the manifest's `case_name` and `case_name_source` columns.
- **Descriptor**: the amendment number for amended complaints, otherwise the ECF document number.
- Files sort by case, and the name matches the `case_id` used in the extraction tables. If two files would get the same name, a short content code is added, so no file overwrites another.

In [ ]:
def safe_name(text, limit=80):
    return re.sub(r"[^\w.-]+", "_", text).strip("_")[:limit]


def slugify(text, limit=40):
    text = re.sub(r"[^\w]+", "_", str(text).strip().lower())
    return re.sub(r"_+", "_", text).strip("_")[:limit].strip("_")


GENERIC_WORDS = {"of", "the", "city", "county", "state", "town", "village", "township", "borough", "inc", "llc",
                 "et", "al"}

def short_case_name(case):
    """'Doe v. County Sheriff et al' -> 'doe_v_county_sheriff';
    'MOSS, JORDAN vs. THE CITY OF RIVERTON et al' -> 'moss_v_riverton'."""
    parts = re.split(r"\s+vs?\.?\s+", case, maxsplit=1, flags=re.I)
    def words(p):
        p = re.sub(r"\bet al\.?", "", p, flags=re.I).split(",")[0]     # "MOSS, JORDAN" -> "MOSS"
        return re.findall(r"[A-Za-z0-9']+", p)
    pw = words(parts[0])
    plaintiff = pw[-1] if pw else ""                                    # surname, e.g. "Maria DeLeon" -> DeLeon
    defendant = ""
    if len(parts) > 1:
        dw = words(parts[1])
        dw = [w for w in dw if w.lower() not in GENERIC_WORDS] or dw
        defendant = "_".join(dw[:2])
    return slugify(f"{plaintiff}_v_{defendant}" if defendant else plaintiff, 40)


# ---- Case names: provider citation, complaint caption, optional LLM, file name -------------------------
# Download file names are often truncated or generic ("14.pdf", "Poe v. City of Springfield e"), so the case name
# is taken, in order of reliability, from: (1) the legal-database citation line in the original download
# ("Poe v. City of Springfield et al, Docket No. ..."), (2) the complaint's caption, read with simple rules,
# (3) optionally an LLM reading the caption, accepted only if both party names appear in the page text,
# (4) the download's file name (marked, to check by hand).

CAPTION_SKIP_RE = re.compile(
    r"^(case\s+\d|document\s+\d|page\s+\d|index\s+no|received nyscef|filed:|nyscef doc|docket no|civil action|"
    r"case no|no\.\s*\d|civil no|jury trial|demand for jury|hon\.|judge|magistrate|u\.s\.d\.j|"
    r"united states district court|district court|(eastern|western|northern|southern|middle|central)\s+district|"
    r"district of|for the|supreme court|superior court|court of|law division|"
    r"in the |[-_=x)(:\s]+$|\d+$|(verified |amended |first amended |second amended |civil rights |class action )*"
    r"complaint\b|.*\bjury (trial|demand)|demand for|civil action|\(?jury\)?)", re.I)
ABOVE_CAPTION_RE = re.compile(r"^(county of|state of [a-z ]+$|[a-z ]+ county\s*:?)|attorney|esq\b|law offices?|counsel|"
                              r"suite|street|drive|avenue|telephone|e-?mail|@|bar (id|no)|p\.?\s?o\.? box", re.I)
PLAINTIFF_LABEL_RE = re.compile(r"\bplaintiffs?(?:\s*/\s*petitioners?)?\s*[,.;:]?\s*$", re.I)
DEFENDANT_LABEL_RE = re.compile(r"\bdefendants?(?:\s*/\s*respondents?)?\s*[,.;:]?\s*$", re.I)
SEPARATOR_RE = re.compile(r"^\s*[-–]*\s*(v\.?|vs\.?|versus|against)\s*[-–]*\s*$", re.I)
CAPACITY_RE = re.compile(r",?\s*(an individual|individually|in (his|her|their) (official|individual|personal)[^,;]*|"
                         r"et al\.?|a municipal(ity| corporation)[^,;]*|a [a-z ]*corporation[^,;]*|"
                         r"a political subdivision[^,;]*|a public entity[^,;]*)", re.I)
ROLE_PREFIX_RE = re.compile(r"^(defendants?|plaintiffs?|petitioners?|respondents?)\s+", re.I)
NOT_A_NAME_RE = re.compile(r"\b(complaint|jury|demand|filed|clerk|division|hereby|alleges?|action|u\.?s\.?c|§|count|"
                           r"unlawful|seizure|damages|docket|plaintiffs?|defendants?|verified|summons|index|exhibit|"
                           r"court|case|page|document)\b", re.I)
KEEP_UPPER = {"NYPD", "LAPD", "LVMPD", "NJ", "NY", "MTA", "USA", "US", "U.S.", "FBI", "DEA", "ICE", "CBP", "TSA", "LLC",
              "INC", "INC.", "II", "III", "IV", "PD", "DC", "D.C.", "SWAT", "CHP", "NJSP", "MBTA", "BART", "DHS"}
SMALL_WORDS = {"of", "the", "and", "for", "a", "an", "in", "on", "at", "to", "de", "la", "del", "van", "von"}


def nice_case(s):
    """'CITY OF NEW YORK' -> 'City of New York', keeping acronyms (NYPD) and mixed-case names (LaRue) as written."""
    words = []
    for k, w in enumerate(s.split()):
        core = w.strip(",.;:()")
        if core.upper() in KEEP_UPPER or (core.isupper() and len(core) <= 4 and not re.search(r"[AEIOUY]", core)):
            words.append(w if core.upper() not in KEEP_UPPER else w.replace(core, core.upper()))
        elif re.fullmatch(r"(Mc|Mac|La|Le|De|Di|Du|Van|O')([A-Z]{2,})[,.;:]?", w):      # "LaRUE" -> "LaRue"
            m = re.fullmatch(r"(Mc|Mac|La|Le|De|Di|Du|Van|O')([A-Z]+)([,.;:]?)", w)
            words.append(m.group(1) + m.group(2).capitalize() + m.group(3))
        elif w.isupper() or w.islower() and k == 0:
            lw = w.lower()
            words.append(lw if (lw.strip(",.") in SMALL_WORDS and k > 0) else
                         "-".join(p[:1].upper() + p[1:] for p in lw.split("-")))
        else:
            words.append(w)
    out = " ".join(words)
    out = re.sub(r"\bMc([a-z])", lambda m: "Mc" + m.group(1).upper(), out)
    out = re.sub(r"\bO'([a-z])", lambda m: "O'" + m.group(1).upper(), out)
    return out


def clean_party(text):
    """First party named in a caption block, and whether more than one party is named."""
    text = re.sub(r"\s+", " ", text).strip(" ,;.")
    text = re.sub(r"\([^)]*\)?", "", text)
    many = bool(re.search(r";|\bet al\b|\s+and\s+[A-Z]| & ", text)) or text.count(",") >= 2
    first = re.split(r";|\s+and\s+(?=[A-Z])|\s+&\s+", text)[0]
    first = re.split(r",?\s+(?:and\s+)?as\s+", first)[0]                     # "X as Administratrix of ...", "and as mother of"
    first = CAPACITY_RE.sub("", first)
    first = ROLE_PREFIX_RE.sub("", first.strip(" ,;."))
    first = first.split(",")[0].strip(" ,;.")
    return nice_case(first), many


def plausible_case_name(name):
    """Reject results that picked up caption furniture or body text instead of party names."""
    if not name or " v. " not in name:
        return False
    for side in name.split(" v. ", 1):
        side = re.sub(r"\bet al\.?$", "", side).strip()
        if not (2 <= len(side) <= 70) or NOT_A_NAME_RE.search(side) or re.search(r"\d{3,}", side) \
                or len(re.findall(r"[A-Za-z]{2,}", side)) == 0:
            return False
    return True


def caption_case_name(text):
    """Case name from a caption ('Plaintiff ... v. ... Defendants'): first plaintiff v. first defendant, with 'et al.'
    when more parties are named. Returns '' if no plausible caption is found."""
    lines = [l.strip() for l in text.split("\n")]
    p = next((i for i, l in enumerate(lines[:150]) if PLAINTIFF_LABEL_RE.search(l)
              and not re.search(r"attorney|counsel|on behalf|for the plaintiff|for plaintiff", l, re.I)), None)
    if p is None:
        return ""
    d = next((i for i in range(p + 1, min(p + 80, len(lines))) if DEFENDANT_LABEL_RE.search(lines[i])), None)
    if d is None:
        return ""
    label_text = PLAINTIFF_LABEL_RE.sub("", lines[p]).strip(" ,")
    block = [label_text] if label_text else []
    for i in range(p - 1, max(-1, p - 8), -1):
        l = lines[i]
        if not l or CAPTION_SKIP_RE.search(l) or SEPARATOR_RE.match(l) or ABOVE_CAPTION_RE.search(l):
            if block:
                break
            continue
        block.insert(0, l)
    seg = lines[p + 1:d + 1]
    sep = next((k for k, l in enumerate(seg) if SEPARATOR_RE.match(l)), None)
    seg = seg[sep + 1:] if sep is not None else seg
    seg[-1] = DEFENDANT_LABEL_RE.sub("", seg[-1])
    defendant_text = " ".join(l for l in seg if l and not CAPTION_SKIP_RE.search(l) and not SEPARATOR_RE.match(l))
    if not block or not defendant_text:
        return ""
    pl, pl_many = clean_party(" ".join(block))
    df_, df_many = clean_party(defendant_text)
    pl_many = pl_many or bool(re.search(r"plaintiffs", lines[p], re.I))
    df_many = df_many or bool(re.search(r"defendants", lines[d], re.I))
    name = f"{pl}{' et al.' if pl_many else ''} v. {df_}{' et al.' if df_many else ''}"
    return name if plausible_case_name(name) else ""


def page_texts_for_caption(doc, start_page, pages=2):
    """Full text of the first pages, and the left part of the first page (two-column captions put the parties on
    the left and court details on the right; reading the left column alone keeps them apart)."""
    full = "\n".join(doc[i].get_text() for i in range(start_page, min(start_page + pages, len(doc))))
    if start_page >= len(doc):
        return full, ""
    page = doc[start_page]
    r = page.rect
    left = page.get_text(clip=fitz.Rect(r.x0, r.y0, r.x0 + r.width * 0.55, r.y1))
    return full, left


PROVIDER_CITATION_RE = re.compile(r"^\s*(?P<name>[^\n]{3,200}?\s+v\.?\s+[^\n]{2,200}?),\s*Docket No\.", re.I | re.M)
PROVIDER_CASE_NAME_RE = re.compile(r"Case Name\s*\n\s*(?P<name>[^\n]{3,200}?\s+v\.?\s+[^\n]{2,200})\s*\n", re.I)


def provider_case_name(path, max_pages=60):
    """Case name from a legal-database citation in the original download, e.g.
    'Poe v. City of Springfield et al, Docket No. 3:23-cv-01111 (S.D. Cal. ...)' -> 'Poe v. City of Springfield et al.'"""
    try:
        with fitz.open(path) as doc:
            for i in range(min(max_pages, len(doc))):
                text = doc[i].get_text()
                m = PROVIDER_CASE_NAME_RE.search(text) or PROVIDER_CITATION_RE.search(text)
                if m:
                    name = re.sub(r"\s+", " ", m.group("name")).strip(" ,")
                    name = re.sub(r"\s+vs?\.?\s+", " v. ", name, count=1, flags=re.I)
                    name = re.sub(r",?\s*et al\.?$", " et al.", name).replace(" et al. et al.", " et al.")
                    name = " v. ".join(nice_case(side) for side in name.split(" v. ", 1))
                    return name if plausible_case_name(name) else ""
    except Exception:
        pass
    return ""


def case_name_from_file(name):
    """Last resort: the download's file or folder name, cleaned (often truncated or generic: check by hand)."""
    name = re.sub(r"^\d+\s*-\s*(entry\s+\S+\s*-\s*)?|^entry\s+\S+\s*-\s*", "", str(name), flags=re.I).strip()
    name = re.sub(r"^\d+_\d+-\w+-\d+_", "", name)                            # "3_25-cv-00001_Johnson ..."
    name = re.sub(r",?\s*Dock(et)?.*$", "", name).strip(" ,")
    if not re.search(r"\s+vs?\.?\s+", name, re.I):
        return ""
    name = re.sub(r"\s+vs?\.?\s+", " v. ", name, count=1, flags=re.I)
    name = re.sub(r",?\s*et al\.?$", " et al.", name)
    return " v. ".join(nice_case(side) for side in name.split(" v. ", 1))


# ---- Optional: LLM fallback for captions the rules cannot read ----------------------------------------
LLM_CASE_NAME_PROMPT = """Below is the first page of a court complaint. From its caption, list the plaintiffs and the
defendants exactly as written (names only, without descriptions such as "an individual" or "in his official capacity").
Return JSON: {"plaintiffs": ["..."], "defendants": ["..."]}. If there is no caption, return empty lists.

PAGE:
"""


def llm_case_name(text, client, model):
    """Ask an LLM for the parties in the caption; accept the answer only if the first plaintiff and the first
    defendant both appear in the page text (the same verification idea as the extraction pipeline's citation check)."""
    import json as _json
    try:
        resp = client.chat.completions.create(
            model=model, temperature=0, response_format={"type": "json_object"},
            messages=[{"role": "user", "content": LLM_CASE_NAME_PROMPT + text[:6000]}])
        data = _json.loads(resp.choices[0].message.content)
    except Exception as e:
        tqdm.write(f"⚠ LLM case name failed: {str(e)[:80]}")
        return ""
    pls, dfs = data.get("plaintiffs") or [], data.get("defendants") or []
    if not pls or not dfs:
        return ""
    flat = re.sub(r"\s+", " ", text).lower()
    if not all(re.sub(r"\s+", " ", x).lower().strip(" ,.") in flat for x in (pls[0], dfs[0])):
        return ""                                                           # not in the text: rejected
    pl, _ = clean_party(pls[0])
    df_, _ = clean_party(dfs[0])
    name = f"{pl}{' et al.' if len(pls) > 1 else ''} v. {df_}{' et al.' if len(dfs) > 1 else ''}"
    return name if plausible_case_name(name) else ""


def best_case_name(original_path, read_path, start_page, llm_client=None, llm_model=None, llm_cache=None, key=None):
    """(case name, source): provider citation, caption (full text, then left column), LLM (optional), else ('', '')."""
    name = provider_case_name(original_path)
    if name:
        return name, "provider citation"
    try:
        with fitz.open(read_path) as doc:
            full, left = page_texts_for_caption(doc, start_page)
    except Exception:
        return "", ""
    for text in (full, left):
        name = caption_case_name(text)
        if name:
            return name, "caption"
    if llm_client is not None:
        if llm_cache is not None and key in llm_cache:
            name = llm_cache[key]
        else:
            name = llm_case_name(full, llm_client, llm_model)
            if llm_cache is not None:
                llm_cache[key] = name
        if name:
            return name, "caption (LLM, verified)"
    return "", ""


def get_llm_client():
    """OpenAI client for the optional case-name fallback (key from OPENAI_API_KEY or config.json)."""
    import json as _json
    from openai import OpenAI
    key = os.environ.get("OPENAI_API_KEY")
    if not key and OPENAI_CONFIG.exists():
        key = _json.loads(OPENAI_CONFIG.read_text()).get("OPENAI_API_KEY")
    if not key:
        print("⚠ No OpenAI API key found: case names are read without the LLM fallback")
        return None
    return OpenAI(api_key=key)


def caption_from_pdf(path, start_page=0, pages=2):
    """Case name from the caption on the document's first pages (from start_page, 0-based); '' if not found."""
    try:
        with fitz.open(path) as doc:
            text = "\n".join(doc[i].get_text() for i in range(start_page, min(start_page + pages, len(doc))))
    except Exception:
        return ""
    return caption_case_name(text)


def standard_name(r):
    """{case_number}_{short_case_name}_{document_type}_{descriptor}.pdf"""
    case_no = r["case_number"] if isinstance(r["case_number"], str) and r["case_number"] else "nocase"
    doc_type = slugify(r["doc_category"] if isinstance(r["doc_category"], str) and r["doc_category"]
                       else r["doc_type"])
    if r["doc_category"] == "amended complaint" and pd.notna(r["amendment_level"]):
        descriptor = str(int(r["amendment_level"]))
    elif isinstance(r["ecf_document"], str) and r["ecf_document"]:
        descriptor = "ecf" + r["ecf_document"].replace("-", "_")
    else:
        descriptor = ""
    # the case's name from its complaint's caption when available, otherwise the download's file/folder name
    name = r["case_name"] if isinstance(r.get("case_name"), str) and r["case_name"] else r["case"]
    parts = [slugify(case_no, 20), short_case_name(name), doc_type, descriptor]
    return "_".join(p for p in parts if p)

## Sorting, the Manifest and Corrections

`separate_complaints` runs every step and copies the files into:

```
complaints_sorted/
├── complaints/                 one complaint per case
│   └── other_versions/         other versions of the complaint (e.g. the original before an amendment)
├── other_documents/            exhibits, cover sheets, orders, motions, …
│   └── duplicates/
├── needs_review/               uncertain cases and scans without a text layer
└── manifest.csv                one row per PDF, with the classification, reasons and new name
```

**Corrections.** The first three columns of `manifest.csv` are for you:

| Column | Use |
|---|---|
| `override_category` | the correct type, e.g. `complaint`, `amended complaint`, `second amended complaint`, `petition`, `motion`, `exhibit`, `civil cover sheet`, `order or judgment`, `other`, `needs review`. Any category containing "complaint" or "petition" counts as a complaint (except e.g. "civil cover sheet", "motion…"), and the amendment level is read from it, so "second amended complaint" is preferred over "amended complaint" as the primary complaint |
| `override_primary` | `yes` to make this the case's primary complaint; `no` to exclude it (only used for complaints; leave blank for other documents) |
| `override_pages` | the complaint's page range in the file, e.g. `8-45`, when the detected `complaint_pages` is wrong or missing; `8-` for page 8 to the end of the file; `8-8` for a single page; `all` to copy the whole file. Only the given pages are copied in Step 3 |
| `override_note` | why (kept in the manifest as documentation) |

When the notebook runs again, it reads your corrections from the existing manifest and applies them before sorting. Corrections are matched to files by **content**, not by name, so they stay attached to the right document even if files are renamed or moved. Corrected rows show confidence *manual*.

In [ ]:
def separate_complaints(source=SOURCE_DIR, output=OUTPUT_DIR, primary_rule=PRIMARY_RULE, dry_run=True):
    source, output = Path(source).resolve(), Path(output).resolve()
    if output == source or source in output.parents:
        raise ValueError("OUTPUT_DIR must be outside SOURCE_DIR, otherwise sorted copies would be read again as input")
    staging = output / "_unzipped"
    extract_zips(source, staging)

    pdfs = [p for root in (source, staging) if root.exists()
            for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf"]
    print(f"Found {len(pdfs)} PDFs")

    rows = []
    progress = tqdm(sorted(pdfs), desc="Reading PDFs", unit="pdf")
    for path in progress:
        root = staging if staging in path.parents else source
        case, entry, is_copy = case_and_entry(path, root)
        sha = file_hash(path)
        ocr_path = output / "_ocr" / f"{sha}.pdf"           # an OCR'd copy (see the OCR section), if one exists
        read_path = ocr_path if ocr_path.exists() else path
        rel = path.relative_to(root)
        batch = rel.parts[0] if len(rel.parts) > 1 else ""            # e.g. FEB_3-4_CASES
        progress.set_postfix_str(f"{batch or '-'} | {path.name[:45]}")
        try:
            info = look_inside(read_path, classify_pdf(read_path))
            if read_path != path:
                info["reasons"] = "read from OCR copy; " + info["reasons"]
            start = ((int(info["complaint_pages"].split("-")[0]) if isinstance(info.get("complaint_pages"), str)
                      else (info.get("content_starts_page") or 1)) - 1)
            info["caption_case_name"] = caption_from_pdf(read_path, start)
        except Exception as e:                      # one problematic file must not stop the whole job
            info = {"pages": None, "case_number": None, "content_starts_page": None, "ecf_document": None,
                    "title": None, "doc_category": None, "amendment_level": None, "doc_type": "needs_review",
                    "complaint_pages": None, "caption_case_name": "",
                    "confidence": "low", "reasons": f"cannot open PDF (error while reading: {type(e).__name__}: {e})"}
        if info["reasons"].startswith(("cannot open", "no text layer")):
            tqdm.write(f"⚠ {batch}/{path.name}: {info['reasons']}")
        rows.append({"batch": batch, "case": case, "entry": entry, "copy_suffix": is_copy, "file": path.name,
                     "path": str(path), "read_path": str(read_path), "ocr_applied": read_path != path,
                     "sha256": sha, "text_hash": text_fingerprint(read_path), **info})

    df = pd.DataFrame(rows)

    # Corrections from a previous manifest (matched by file content, so renaming or moving does not matter)
    df["override_category"], df["override_primary"], df["override_pages"], df["override_note"] = "", "", "", ""
    manifest_path = output / "manifest.csv"
    if manifest_path.exists():
        old = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
        if "sha256" in old:
            old = old.drop_duplicates("sha256").set_index("sha256")
            for col in ["override_category", "override_primary", "override_pages", "override_note"]:
                if col in old:
                    df[col] = df["sha256"].map(old[col]).fillna("")
    # Documents without a readable case number (e.g. cover sheets) take the number found in the same case
    case_numbers = df.dropna(subset=["case_number"]).groupby("case")["case_number"].agg(lambda v: v.mode().iloc[0])
    df["case_number"] = df["case_number"].fillna(df["case"].map(case_numbers))
    # A case is identified by its case number plus the first plaintiff's surname when the number is known
    # (case numbers repeat across courts, and two cases can share a name across batches), otherwise by its name
    # (files with generic names such as "1 copy 10.pdf" have no case name: they are keyed by case number alone,
    # so a complaint and its cover sheet with the same number are grouped together)
    has_name = df["case"].str.contains(r"\s+vs?\.?\s+", case=False, regex=True)
    surname = df["case"].map(lambda c: short_case_name(c).split("_v_")[0])
    df["case_key"] = (df["case_number"] + "_" + surname).where(has_name, df["case_number"])
    df["case_key"] = df["case_key"].where(df["case_number"].notna(), df["case"])
    # A file whose download name has no case name (e.g. "1 copy 14.pdf") is keyed by its case number alone; if
    # exactly one named case has the same number, the file belongs to that case (otherwise one case would be
    # counted twice, once under each key)
    named = df[has_name & df["case_number"].notna()].groupby("case_number")["case_key"].unique()
    single = {num: keys[0] for num, keys in named.items() if len(keys) == 1}
    generic = ~has_name & df["case_number"].notna()
    df.loc[generic, "case_key"] = df.loc[generic, "case_number"].map(single).fillna(df.loc[generic, "case_key"])

    n_overrides = 0
    for idx, r in df.iterrows():
        cat = str(r["override_category"]).strip().lower()
        if cat:
            n_overrides += 1
            df.at[idx, "doc_category"] = cat
            # any kind of complaint or petition counts ("second amended complaint", "verified complaint", ...),
            # but not documents about a complaint ("civil cover sheet", "motion to dismiss complaint", ...)
            is_complaint = (re.search(r"\bcomplaint\b|\bpetition\b", cat)
                            and not re.search(r"cover sheet|motion|answer|exhibit|summons|order|opinion|brief", cat))
            df.at[idx, "doc_type"] = ("complaint" if is_complaint
                                      else "needs_review" if cat == "needs review" else "other")
            if is_complaint:
                m = re.search(r"\b(first|second|third|fourth|fifth|\d+(?:st|nd|rd|th))?\s*amended\b", cat)
                df.at[idx, "amendment_level"] = (0 if not m else
                                                 ORDINALS.get((m.group(1) or "first"), None)
                                                 or int(re.sub(r"\D", "", m.group(1)) or 1))
            df.at[idx, "confidence"] = "manual"
            df.at[idx, "reasons"] = f"manual override ({r['override_note'] or 'no note'}); " + str(r["reasons"])
    # Page-range corrections, e.g. "8-45" (complaint pages only) or "all" (copy the whole file)
    n_pages = 0
    for idx, r in df.iterrows():
        pages = str(r["override_pages"]).strip().lower().replace(" ", "").replace("–", "-")
        if not pages:
            continue
        if re.fullmatch(r"\d+-", pages) and pd.notna(r["pages"]):     # "6-" = from page 6 to the end of the file
            pages = f"{pages}{int(r['pages'])}"
        if pages == "all":
            df.at[idx, "complaint_pages"] = None
        elif re.fullmatch(r"\d+-\d+", pages) and int(pages.split("-")[0]) <= int(pages.split("-")[1]) \
                and (pd.isna(r["pages"]) or int(pages.split("-")[1]) <= int(r["pages"])):
            df.at[idx, "complaint_pages"] = pages
        else:
            tqdm.write(f"⚠ ignored override_pages '{r['override_pages']}' for {r['file']} "
                       f"(use e.g. 8-45, 8- for page 8 to the end, 8-8 for a single page, or all)")
            continue
        n_pages += 1
        df.at[idx, "reasons"] = f"pages set manually ({pages}); " + str(df.at[idx, "reasons"])
    if n_overrides or n_pages:
        print(f"Applied {n_overrides} manual override(s) and {n_pages} page-range correction(s) "
              f"from the previous manifest")

    # Duplicates: identical files, or identical text. Keep the copy without a "-2"-style suffix.
    df["duplicate_of"] = None
    df = df.sort_values(["copy_suffix", "path"]).reset_index(drop=True)
    first_seen = {}
    for idx, r in df.iterrows():
        keys = [("file", r["sha256"])]
        if pd.notna(r["text_hash"]):
            keys.append(("text", r["case"], r["text_hash"]))       # same text within the same case
        match = next((first_seen[k] for k in keys if k in first_seen), None)
        if match is not None:
            df.at[idx, "duplicate_of"] = df.at[match, "path"]
        else:
            for k in keys:
                first_seen[k] = idx

    # One primary complaint per case
    df["primary"] = False
    complaints = df[(df["doc_type"] == "complaint") & df["duplicate_of"].isna()]
    for case, group in complaints.groupby("case_key"):
        forced = group[group["override_primary"].astype(str).str.strip().str.lower().isin(["yes", "y", "true", "1"])]
        excluded = group["override_primary"].astype(str).str.strip().str.lower().isin(["no", "n", "false", "0"])
        candidates = group[~excluded]
        if len(forced):
            df.at[forced.index[0], "primary"] = True       # your choice wins
            continue
        if candidates.empty:
            continue
        order = candidates.assign(level=candidates["amendment_level"].fillna(0),
                                  ent=candidates["entry"].fillna(0)).sort_values(["level", "ent"])
        chosen = order.index[-1] if primary_rule == "latest" else order.index[0]
        df.at[chosen, "primary"] = True

    # Destination folder for each file
    def destination(r):
        if pd.notna(r["duplicate_of"]):
            return "other_documents/duplicates"
        if r["doc_type"] == "complaint":
            return "complaints" if r["primary"] else "complaints/other_versions"
        return "needs_review" if r["doc_type"] == "needs_review" else "other_documents"
    df["destination"] = df.apply(destination, axis=1)

    # Case names (used in the file names and the manifest). For each case's primary complaint, in order of
    # reliability: the legal-database citation in the original download, the complaint's caption (rules), the
    # optional LLM reading of the caption (verified against the page text), else the download's file name.
    import json as _json
    llm_client = get_llm_client() if USE_LLM_FOR_CASE_NAMES else None
    cache_path = output / "_case_names_llm.json"
    llm_cache = _json.loads(cache_path.read_text()) if cache_path.exists() else {}
    case_names = {}
    primaries = df[df["primary"]]
    for _, r in tqdm(primaries.iterrows(), total=len(primaries), desc="Reading case names", unit="case"):
        start = (int(r["complaint_pages"].split("-")[0]) if isinstance(r["complaint_pages"], str)
                 else int(r["content_starts_page"]) if pd.notna(r["content_starts_page"]) else 1) - 1
        name, source = best_case_name(r["path"], r["read_path"], start, llm_client, LLM_MODEL, llm_cache,
                                      key=f"{r['sha256']}:{start}")
        case_names[r["case_key"]] = (name, source)
    if llm_client is not None:
        cache_path.write_text(_json.dumps(llm_cache))
    # cases without a primary complaint: a caption from any of their documents
    for key, g in df[~df["case_key"].isin(case_names)].groupby("case_key"):
        caps = [c for c in g["caption_case_name"].fillna("") if c]
        case_names[key] = (caps[0], "caption") if caps else ("", "")
    df["case_name"] = df["case_key"].map(lambda k: case_names.get(k, ("", ""))[0])
    df["case_name_source"] = df["case_key"].map(lambda k: case_names.get(k, ("", ""))[1])
    no_name = df["case_name"] == ""
    df.loc[no_name, "case_name"] = df.loc[no_name, "case"].map(case_name_from_file)
    df.loc[no_name & (df["case_name"] != ""), "case_name_source"] = "file name (check)"
    print("Case names by source (primary complaints):")
    print(df[df["primary"]]["case_name_source"].replace("", "none").value_counts().to_string())

    # Standard file names; a short content code is added only if two files would get the same name
    names, used = [], set()
    for _, r in df.iterrows():
        base = standard_name(r)
        name = base if (r["destination"], base) not in used else f"{base}_{r['sha256'][:8]}"
        used.add((r["destination"], name))
        names.append(name + ".pdf")
    df["new_name"] = names

    if not dry_run:
        # the sorted folders are generated output: clear them so renamed files do not leave stale copies
        for sub in ["complaints", "other_documents", "needs_review"]:
            shutil.rmtree(output / sub, ignore_errors=True)
        for _, r in tqdm(df.iterrows(), total=len(df), desc="Copying files", unit="file"):
            target_dir = output / r["destination"]
            target_dir.mkdir(parents=True, exist_ok=True)
            pages = r.get("complaint_pages")
            if isinstance(pages, str) and pages and r["doc_type"] == "complaint":
                # a complaint inside a larger file: copy only its pages
                a, b = (int(x) for x in pages.split("-"))
                with fitz.open(r["read_path"]) as src_doc:
                    part = fitz.open()
                    part.insert_pdf(src_doc, from_page=a - 1, to_page=b - 1)
                    part.save(target_dir / r["new_name"])
            else:
                shutil.copy2(r["read_path"], target_dir / r["new_name"])     # the OCR'd copy when there is one

    output.mkdir(parents=True, exist_ok=True)
    manifest = output / "manifest.csv"
    first = ["override_category", "override_primary", "override_pages", "override_note", "batch", "case", "case_name", "case_name_source",
             "case_number", "new_name",
             "doc_category", "doc_type", "confidence", "primary", "destination", "title", "complaint_pages", "reasons"]
    df = df[first + [c for c in df.columns if c not in first and c != "text_hash"]]
    df.to_csv(manifest, index=False, encoding="utf-8-sig")

    # Summary
    n_cases = df["case_key"].nunique()
    n_with = df.loc[df["primary"], "case_key"].nunique()
    print(f"\nCases: {n_cases} | with a primary complaint: {n_with} | without: {n_cases - n_with}")
    print(df["destination"].value_counts().to_string())
    missing = sorted(set(df["case_key"]) - set(df.loc[df["primary"], "case_key"]))
    if missing:
        print("\nCases without a complaint found (check needs_review/):")
        for c in missing:
            print("  -", c, "|", "; ".join(sorted(set(df.loc[df["case_key"] == c, "case"]))))
    print(f"\nManifest: {manifest}")
    return df

### Before Step 1: check that your corrections are loaded

The dry run reads your corrections from `manifest.csv` in the output folder. This cell shows what it will find; with your current manifest: 664 rows, 311 category corrections, 93 page ranges and 288 primary choices. If it shows zeros, the corrected file is missing or has a different name (e.g. `manifest (1).csv`).

In [ ]:
mf = OUTPUT_DIR / "manifest.csv"
x = pd.read_csv(mf, dtype=str, keep_default_na=False)
print("found:", mf.exists(), "| rows:", len(x))
print("category corrections:", (x["override_category"] != "").sum(),
      "| page ranges:", (x["override_pages"] != "").sum(),
      "| primary choices:", x["override_primary"].isin(["yes", "no"]).sum())

## Step 1: Dry Run

Classify every PDF and write the manifest, without copying any files.

A progress bar shows the count of PDFs read, the current batch and file, the rate, and the estimated time remaining. Files that cannot be read (damaged, or scans without text) are flagged above the bar as they occur. With several hundred PDFs on Google Drive this can take a while. If Colab disconnects, simply run it again: nothing is copied during the dry run.

Next: the OCR check (Step O1), then the review (Step 2).

In [ ]:
manifest_df = separate_complaints(SOURCE_DIR, OUTPUT_DIR, PRIMARY_RULE, dry_run=True)

## OCR for Scanned Documents (before reviewing)

Run this section **before** the review in Step 2: scans that OCR makes readable are then classified automatically, so they no longer appear among the documents to review.

**Is it necessary?** Only if some complaints are scans. The extraction pipeline reads a PDF's text layer; a scanned complaint without one yields no text, so every field would come back "not stated", and the case would silently drop out of the dataset. Run the check below first: if it finds no scanned complaints, skip Steps O2 and O3 and go to Step 2.

The check counts, for every complaint and every document needing review, the share of pages with readable text. A low share catches fully scanned documents, and also provider PDFs whose title page or docket report has text but whose complaint pages are scanned.

| text_share | Meaning | Action |
|---|---|---|
| 0 | fully scanned | OCR |
| low (e.g. 0.05) | text title page or docket report, scanned complaint | OCR |
| 0.5–0.8 | partly scanned, often scanned exhibits after a typed complaint | OCR if the complaint pages are scanned |
| above 0.8 | readable | none |

In [ ]:
def text_coverage(path, min_chars=50):
    """Number of pages, and number of pages with readable text (at least min_chars characters)."""
    with fitz.open(path) as doc:
        return len(doc), sum(len(page.get_text().strip()) >= min_chars for page in doc)


def check_text_layers(manifest, threshold=0.8):
    """Share of pages with text for every complaint and every document needing review.
    Uses the OCR'd copy when one exists. Returns all checked documents, and those below the threshold."""
    candidates = manifest[manifest["doc_type"].isin(["complaint", "needs_review"])
                          & manifest["duplicate_of"].isna()]
    rows = []
    for _, r in tqdm(candidates.iterrows(), total=len(candidates), desc="Checking text layers", unit="pdf"):
        try:
            pages, with_text = text_coverage(r["read_path"])
        except Exception as e:
            tqdm.write(f"⚠ cannot open {r['file']}: {e}")
            pages, with_text = 0, 0
        rows.append({"batch": r["batch"], "case": r["case"], "file": r["file"], "doc_type": r["doc_type"],
                     "primary": r["primary"], "ocr_applied": r["ocr_applied"], "pages": pages,
                     "pages_with_text": with_text, "text_share": round(with_text / pages, 2) if pages else 0.0,
                     "path": r["path"], "sha256": r["sha256"]})
    coverage = pd.DataFrame(rows).sort_values("text_share").reset_index(drop=True)
    return coverage, coverage[coverage["text_share"] < threshold]


def run_ocr(documents, output, language="eng", redo=False):
    """Run OCRmyPDF on the given documents. Copies are written to <output>/_ocr/<content hash>.pdf;
    the originals are never changed. The next dry run reads (and Step 3 copies) the OCR'd version.
    redo=False: only pages without text are OCR'd (provider title pages and typed pages are kept as they are).
    redo=True: existing poor OCR is replaced (for garbled text layers)."""
    import subprocess, sys
    ocr_dir = Path(output) / "_ocr"
    ocr_dir.mkdir(parents=True, exist_ok=True)
    done, failed = 0, []
    for _, r in tqdm(documents.iterrows(), total=len(documents), desc="Running OCR", unit="pdf"):
        target = ocr_dir / f"{r['sha256']}.pdf"
        if target.exists() and not redo:
            continue
        cmd = [sys.executable, "-m", "ocrmypdf", "--quiet", "--language", language,
               "--redo-ocr" if redo else "--skip-text",
               str(r["path"]), str(target)]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode == 0 and target.exists():
            done += 1
        else:
            target.unlink(missing_ok=True)
            failed.append(r["file"])
            tqdm.write(f"⚠ OCR failed for {r['file']}: {(proc.stderr or '').strip()[-200:]}")
    print(f"OCR finished: {done} document(s) processed, {len(failed)} failed")
    return failed

### Step O1: Check text layers

In [ ]:
coverage, needs_ocr = check_text_layers(manifest_df, threshold=0.8)
print(f"{len(needs_ocr)} of {len(coverage)} checked documents have text on fewer than 80% of their pages")
print(f"of which primary complaints: {int(needs_ocr['primary'].sum())}")
display(needs_ocr.drop(columns=["path", "sha256"]))

In [ ]:
# Primary complaints of any origin (including your manual corrections) with too little text
primary_cov, primary_low = check_text_layers(manifest_df[manifest_df["primary"]], threshold=0.8)
print(f"Primary complaints below 80% text: {len(primary_low)}")
display(primary_low.drop(columns=["path", "sha256"]))
# to OCR them as well:  failed = run_ocr(primary_low, OUTPUT_DIR)

### Step O2: Run OCR

OCRmyPDF adds a text layer to scanned pages while keeping their appearance. OCR'd copies are saved in `COMPLAINTS_SORTED/_ocr/`, named by the original file's content hash; the originals in `COMPLAINTS_ALL` are not changed. The first cell installs OCRmyPDF; OCR takes a few seconds per page, so the next cell runs a test on two documents. (The `ocrmypdf` package from apt is avoided on purpose: it can clash with Colab's Python libraries.) For documents in another language, set `language`, e.g. `"spa"` for Spanish (after `!apt-get install -qq tesseract-ocr-spa`).

- `redo=False` (default) processes only pages without text.
- `redo=True` replaces an existing but poor text layer (e.g., garbled text such as "pennanent" for "permanent"); use it for specific documents whose extraction shows many failed quote checks.

In [ ]:
# OCRmyPDF (Python package) with its OCR engine (Tesseract) and Ghostscript
!apt-get install -qq tesseract-ocr ghostscript > /dev/null
!pip install -q ocrmypdf
!python -m ocrmypdf --version

In [ ]:
# Test on two documents first, then open their OCR'd copies in COMPLAINTS_SORTED/_ocr/ to check the result
failed = run_ocr(needs_ocr.head(2), OUTPUT_DIR)

In [ ]:
# Then run on all documents that need OCR (already processed ones are skipped)
failed = run_ocr(needs_ocr, OUTPUT_DIR)

### Step O3: Re-run the dry run

Classify again, now reading the OCR'd copies: documents with one show `ocr_applied` = True, and their `reasons` start with "read from OCR copy". Your earlier corrections in `manifest.csv` are kept and still take precedence. Step 3 will copy the OCR'd versions into the sorted folders.

In [ ]:
manifest_df = separate_complaints(SOURCE_DIR, OUTPUT_DIR, PRIMARY_RULE, dry_run=True)

# confirm: the OCR'd documents should now have a high text share
coverage, needs_ocr = check_text_layers(manifest_df, threshold=0.8)
print(f"Still below 80% text after OCR: {len(needs_ocr)} document(s)")
display(needs_ocr.drop(columns=["path", "sha256"]))

## Step 2: Review the Classification

After OCR (if any), check the rows most likely to be wrong: every *needs review* row, complaints with *medium* confidence, unidentified documents, and cases where no complaint was found.

In [ ]:
pd.set_option("display.max_colwidth", 100)
cols = ["batch", "case", "case_number", "file", "title", "doc_category", "doc_type", "confidence", "primary",
        "complaint_pages", "pages", "destination", "new_name", "reasons"]

print("NEEDS REVIEW")
display(manifest_df.loc[manifest_df["doc_type"] == "needs_review", cols])

In [ ]:
print("COMPLAINTS WITH MEDIUM CONFIDENCE, AND UNIDENTIFIED DOCUMENTS")
display(manifest_df.loc[((manifest_df["doc_type"] == "complaint") & (manifest_df["confidence"] == "medium"))
                        | (manifest_df["doc_category"] == "unidentified"), cols])

### Spot-check page ranges of complaints found inside other documents

For a sample of complaints found inside summons filings, notices of removal and bundles, this prints the opening lines of the pages just before, at, and after each end of the detected range. A correct range starts on the complaint's caption page and ends on its last page (prayer for relief, jury demand or signature); the page before it and the page after it belong to something else. Correct a wrong range in `override_pages` (e.g. `8-45`).

In [ ]:
HEADER_LINE_RE = re.compile(r"^(INDEX NO\.|RECEIVED NYSCEF|FILED:|NYSCEF DOC\. NO\.|Case \d+:\d+-|"
                            r".*Docket No\..*\(.*\d{4}\)|\d{1,3}$|[A-Z]?\s*NO\. \d+/\d{4}$|INDEX$)")

def first_lines(doc, page_no, n=4):
    """Opening lines of a page (1-based), skipping court/provider header lines repeated on every page
    (NYSCEF stamps, ECF headers, provider citations, page numbers)."""
    if page_no < 1 or page_no > len(doc):
        return "   (no page: start or end of file)"
    lines = [l.strip() for l in doc[page_no - 1].get_text().split("\n")
             if l.strip() and not HEADER_LINE_RE.match(l.strip())][:n]
    return "\n".join("   " + l[:100] for l in lines) or "   (no text)"

inside = manifest_df[manifest_df["reasons"].str.contains("found inside", na=False) & manifest_df["primary"]]
print(len(inside), "primary complaints found inside other documents; checking up to 15\n")
for _, r in inside.sample(min(15, len(inside)), random_state=1).iterrows():
    start, end = (int(x) for x in r["complaint_pages"].split("-"))
    with fitz.open(r["read_path"]) as doc:
        print("=" * 100)
        print(f"{r['case']}  |  pages {start}-{end} of {len(doc)}  |  title: {r['title']}")
        print(f"\n-- page {start - 1} (should NOT be the complaint):\n{first_lines(doc, start - 1)}")
        print(f"\n-- page {start} (should be the complaint's FIRST page):\n{first_lines(doc, start)}")
        print(f"\n-- page {end} (should be the complaint's LAST page):\n{first_lines(doc, end)}")
        print(f"\n-- page {end + 1} (should NOT be the complaint):\n{first_lines(doc, end + 1)}\n")

In [ ]:
print("CASES WITH SEVERAL COMPLAINT VERSIONS (check which one is primary)")
# duplicates (the same file saved twice) are left out: only genuinely different versions are shown
versions = (manifest_df[(manifest_df["doc_type"] == "complaint") & manifest_df["duplicate_of"].isna()]
            .groupby("case_key").filter(lambda g: len(g) > 1))
print(f"{versions['case_key'].nunique()} case(s) with more than one complaint version")
display(versions.sort_values(["case_key", "amendment_level", "entry"])[
    ["case_key", "file", "title", "doc_category", "amendment_level", "primary", "complaint_pages", "pages", "confidence"]])

In [ ]:
print("CASES WITHOUT A COMPLAINT")
missing = sorted(set(manifest_df["case_key"]) - set(manifest_df.loc[manifest_df["primary"], "case_key"]))
display(manifest_df.loc[manifest_df["case_key"].isin(missing), cols])

**To correct a row:** open `manifest.csv` (in the output folder) in Excel or Google Sheets and fill in the override columns: `override_category` (e.g. `complaint`, `first amended complaint`, `motion`, `exhibit`), `override_primary` (`yes`/`no`), `override_pages` (e.g. `8-45`, `8-` to the end of the file, `8-8`, or `all`) and `override_note` (the reason, e.g. `range fixed: pp. 27-28 are affirmations`). Page numbers are the PDF's own page numbers (page 1 = first page of the file). Put page ranges in `override_pages`, not in the note: notes are documentation only.

**Saving:** Google Sheets does not change the CSV file itself. Use File → Download → Comma-separated values, rename the download to exactly `manifest.csv`, and upload it to the output folder, replacing the old file. Then **re-run Step 1**: it reports how many corrections were applied (e.g. *"Applied 292 manual override(s) and 38 page-range correction(s)"*) and warns about any page range it could not use. Repeat until the review tables look right.

**Observation:** _(How many cases have a primary complaint? Which files needed corrections, and why?)_

## Step 3: Copy the Files

Once the classification looks right, run the sort for real. The sorted folders are regenerated on each run (your corrections in the manifest are kept).

In [ ]:
manifest_df = separate_complaints(SOURCE_DIR, OUTPUT_DIR, PRIMARY_RULE, dry_run=False)

### Check the copied complaints

The `complaints` folder should contain one PDF per case with a complaint, and every complaint copied from a page range should have exactly that many pages. Then open a few copies, e.g. one from a New York summons filing, one from a notice of removal, and one with a corrected page range, and confirm each starts on the complaint's first page and contains the whole complaint.

In [ ]:
comp_dir = OUTPUT_DIR / "complaints"
files = sorted(comp_dir.glob("*.pdf"))
prim = manifest_df[manifest_df["primary"]].set_index("new_name")
print("PDFs in complaints/:", len(files), "| cases with a primary complaint:", manifest_df.loc[manifest_df["primary"], "case_key"].nunique())

mismatch = []
for f in files:
    if f.name not in prim.index or not isinstance(prim.loc[f.name, "complaint_pages"], str):
        continue
    a, b = (int(x) for x in prim.loc[f.name, "complaint_pages"].split("-"))
    with fitz.open(f) as d:
        if len(d) != b - a + 1:
            mismatch.append((f.name, prim.loc[f.name, "complaint_pages"], len(d)))
print("page-count mismatches:", len(mismatch))
for x in mismatch:
    print("  ", x)

### Step 3b: Screen the Complaints for Wrong Documents

A selected "complaint" can turn out to be another filing, e.g. a magistrate judge's recommendation or an order (in the validation sample, 2 of 21 selected files were not the complaint). Extraction on such a file gives meaningless values, so screen the copied complaints **before the extraction run**.

This cell reads the first two pages of every copied complaint (no API cost, a minute or two) and flags files that look like another filing (recommendation, order, opinion, motion, notice of removal, answer, judgment, stipulation, settlement) or never mention a complaint or petition. It writes **`flagged_documents.csv`** in the output folder; it is never overwritten once it exists. The screen is deliberately broad: some flagged files are real complaints.

In [ ]:
# ---- Step 3b: screen the copied complaints for documents that are not complaints ----
NOT_COMPLAINT_RE = re.compile(
    r"REPORT AND RECOMMENDATION|RECOMMENDATION OF (THE )?UNITED STATES MAGISTRATE|"
    r"MEMORANDUM (OPINION|AND ORDER|DECISION)|OPINION AND ORDER|DECISION AND ORDER|"
    r"ORDER (GRANTING|DENYING|DISMISSING|ON )|NOTICE OF REMOVAL|MOTION (TO|FOR) |"
    r"ANSWER TO|FINAL JUDGMENT|STIPULATION|SETTLEMENT AGREEMENT", re.I)
COMPLAINT_WORD_RE = re.compile(r"COMPLAINT|PETITION", re.I)

screen_path = OUTPUT_DIR / "flagged_documents.csv"
if screen_path.exists():
    print("Screening sheet already exists and was NOT overwritten (your decisions are safe):", screen_path)
    flagged = pd.read_csv(screen_path, dtype=str, keep_default_na=False)
else:
    man = pd.read_csv(OUTPUT_DIR / "manifest.csv", dtype=str, keep_default_na=False)
    prim = man[man["primary"].str.upper() == "TRUE"]
    rows = []
    for _, r in tqdm(prim.iterrows(), total=len(prim), desc="Screening complaints", unit="pdf"):
        f = OUTPUT_DIR / "complaints" / r["new_name"]
        if not f.exists():
            continue
        with fitz.open(f) as d:
            head = " ".join(d[i].get_text() for i in range(min(2, len(d))))[:4000]
        m = NOT_COMPLAINT_RE.search(head)
        if m or not COMPLAINT_WORD_RE.search(head):
            rows.append({"case": r["case"], "complaint_file": r["new_name"], "original_file": r["file"],
                         "sha256": r["sha256"],
                         "reason": m.group(0) if m else "no 'complaint' or 'petition' on pp. 1-2",
                         "first_lines": " | ".join(l.strip() for l in head.splitlines() if l.strip())[:250],
                         "is_complaint": "", "correct_category": "", "note": ""})
    flagged = pd.DataFrame(rows, columns=["case", "complaint_file", "original_file", "sha256", "reason", "first_lines",
                                          "is_complaint", "correct_category", "note"])
    flagged.to_csv(screen_path, index=False, encoding="utf-8-sig")
    print(f"{len(flagged)} of {len(prim)} complaints flagged for checking -> {screen_path}")
display(flagged[["case", "complaint_file", "reason", "first_lines"]])

**Triage the undecided rows** (optional, saves time). Most flags are complaints that don't start on page 1 or 2 (a title sheet, docket page or notice of removal comes first). For every row with an empty `is_complaint`, this cell looks for a complaint title in the first 20 pages and writes a suggestion into the sheet (columns `complaint_starts_page` and `suggestion`); rows you already decided are not changed. Then:
- **"complaint title found on p. N"**: set `is_complaint` to `yes` (open two or three to confirm the suggestion is reliable);
- **"CHECK BY HAND"**: open the file and decide (`yes`, `no`, or the complaint's pages in `complaint_pages`).

In [ ]:
screen_path = OUTPUT_DIR / "flagged_documents.csv"
screen = pd.read_csv(screen_path, dtype=str, keep_default_na=False)
TITLE_RE = re.compile(r"^\s*((VERIFIED|AMENDED|FIRST|SECOND|THIRD|CIVIL RIGHTS|ORIGINAL|CLASS ACTION|PLAINTIFF'?S?'?)\s+)*"
                      r"(COMPLAINT|PETITION)\b", re.I)
WRONG_RE = re.compile(r"MOTION|ANSWER|ORDER|RECOMMENDATION|OPINION|JUDGMENT|NOTICE OF REMOVAL", re.I)


def complaint_start(path, max_pages=20):
    """First page (1-based) with a line that reads like a complaint title, or None."""
    with fitz.open(path) as d:
        for i in range(min(max_pages, len(d))):
            for line in d[i].get_text().splitlines():
                line = line.strip()
                if len(line) < 80 and TITLE_RE.match(line) and not WRONG_RE.search(line):
                    return i + 1
    return None


for col in ["complaint_pages", "complaint_starts_page", "suggestion"]:
    if col not in screen:
        screen[col] = ""
todo = (screen["is_complaint"].str.strip() == "") & (screen["complaint_pages"].str.strip() == "")
for i in tqdm(screen.index[todo], desc="Triage", unit="pdf"):
    f = OUTPUT_DIR / "complaints" / screen.at[i, "complaint_file"]
    start = complaint_start(f) if f.exists() else None
    screen.at[i, "complaint_starts_page"] = "" if start is None else str(start)
    screen.at[i, "suggestion"] = (f"complaint title found on p. {start}: probably a complaint" if start
                                  else "no complaint title in the first 20 pages: CHECK BY HAND")
screen.to_csv(screen_path, index=False, encoding="utf-8-sig")
print(f"{int(todo.sum())} undecided rows triaged:")
print(screen.loc[todo, "suggestion"].str.split(":").str[0].value_counts().to_string())
display(screen.loc[todo & screen["suggestion"].str.contains("CHECK"), ["case", "complaint_file", "reason", "first_lines"]])

**Review `flagged_documents.csv`** (open it in Excel or Google Sheets; save it back as CSV under the same name). For each row, open the file in `complaints/` and fill in:

| Column | Enter |
|---|---|
| `is_complaint` | `yes` (it is the complaint: nothing changes) or `no` |
| `correct_category` | if `no`: what it is, e.g. `report and recommendation`, `order or judgment`, `motion`, `answer`, `other`. Avoid the words *complaint* and *petition* here (e.g. write `answer`, not `answer to complaint`): a category containing them is treated as a complaint |
| `complaint_pages` | if the copied file is the wrong part of a file that **does** contain the complaint (e.g. a motion copied instead of the complaint pages): the complaint's pages in the **original** file, e.g. `27-60` or `27-` (to the end). Leave `is_complaint` empty in that case |
| `note` | optional, e.g. `magistrate's recommendation, ECF 34` |

If the case's real complaint is in another file, mark that file in `manifest.csv` as usual (`override_category` = `complaint`, `override_primary` = `yes`, `override_pages` if needed).

### Step 3c: Apply the Screening Decisions

Marks every file you set to `no` (and its duplicate copies) as not a complaint in `manifest.csv`, and sets the corrected page range where you entered `complaint_pages`, with a note (a backup of the manifest is kept as `manifest_before_screening.csv`). Then **re-run Step 1 and Step 3**, so `complaints/` contains only complaints.

In [ ]:
# ---- Step 3c: apply your screening decisions to the manifest ----
screen = pd.read_csv(OUTPUT_DIR / "flagged_documents.csv", dtype=str, keep_default_na=False)
if "complaint_pages" not in screen:
    screen["complaint_pages"] = ""
decision = screen["is_complaint"].str.strip().str.lower()
pages = screen["complaint_pages"].str.strip().str.replace(" ", "")
undecided = int(((decision == "") & (pages == "")).sum())

mpath = OUTPUT_DIR / "manifest.csv"
backup = OUTPUT_DIR / "manifest_before_screening.csv"
if not backup.exists():
    shutil.copy2(mpath, backup)
man = pd.read_csv(mpath, dtype=str, keep_default_na=False)
not_complaint, new_pages, bad = 0, 0, []
for idx, r in screen.iterrows():
    rows = man.index[man["sha256"] == r["sha256"]]                    # the file and any duplicate copies
    if not len(rows):
        continue
    extra = ("; " + r["note"].strip()) if r["note"].strip() else ""
    if pages[idx]:                                                    # the complaint is elsewhere in the same file
        if not re.fullmatch(r"\d+-\d*", pages[idx]):
            bad.append(f"{r['complaint_file']}: '{r['complaint_pages']}' (use e.g. 27-60 or 27-)")
            continue
        for i in rows:
            man.at[i, "override_pages"] = pages[idx]
            man.at[i, "override_note"] = f"screening: complaint on pp. {pages[idx]} of the original file" + extra
        new_pages += 1
    elif decision[idx] == "no":                                       # the file is not a complaint
        category = r["correct_category"].strip() or "other"
        for i in rows:
            man.at[i, "override_category"] = category
            man.at[i, "override_primary"] = ""
            man.at[i, "override_pages"] = ""
            man.at[i, "override_note"] = "screening: not a complaint (" + r["reason"] + ")" + extra
        not_complaint += 1
man.to_csv(mpath, index=False, encoding="utf-8-sig")
print(f"Flagged files: {len(screen)} | undecided: {undecided}")
print(f"Marked as not a complaint: {not_complaint} | complaint page range corrected: {new_pages} "
      f"(backup: {backup.name})")
for b in bad:
    print("⚠ page range not understood:", b)
print("Next: re-run Step 1 and Step 3.")

## Step 4: Final Check of Cases Without a Complaint

The cases still without a complaint are where the automatic rules may have **missed** a complaint, so they are checked by hand, one decision per case.

**4a.** Create a review sheet (one row per case, with empty columns for your decisions).

In [ ]:
m = pd.read_csv(OUTPUT_DIR / "manifest.csv", dtype=str, keep_default_na=False)
m["is_primary"] = m["primary"].str.strip().str.lower() == "true"

missing_keys = sorted(set(m["case_key"]) - set(m.loc[m["is_primary"], "case_key"]))
missing_df = m[m["case_key"].isin(missing_keys) & (m["duplicate_of"] == "")][
    ["case_key", "case", "batch", "file", "pages", "doc_category", "title", "override_note", "path"]
].sort_values(["case_key", "file"]).reset_index(drop=True)

review_cases = (missing_df.groupby("case_key")
                .agg(case=("case", "first"),
                     files=("file", "count"),
                     file_names=("file", lambda f: " | ".join(f)),
                     document_types=("doc_category", lambda c: "; ".join(sorted(set(c)))))
                .reset_index())
for col in ["final_status", "complaint_file", "complaint_pages", "check_note"]:
    review_cases[col] = ""

review_path = OUTPUT_DIR / "cases_without_complaint_review.csv"
if review_path.exists():
    print("A review sheet already exists and was NOT overwritten (your completed review is safe):", review_path)
    print("Delete or rename it only if you want to start a fresh review. Showing the saved sheet:")
    display(pd.read_csv(review_path, dtype=str, keep_default_na=False))
    print(f"\nCases currently without a complaint: {len(review_cases)}")
else:
    review_cases.to_csv(review_path, index=False, encoding="utf-8-sig")
    print(f"{len(review_cases)} cases, {len(missing_df)} files -> {review_path}")
    display(review_cases)

**4b.** Fill in the sheet (open it in Google Sheets). For each case, open its files (names in `file_names`; use the Drive search bar) and look for a page with a caption and a complaint title.

| Column | What to enter |
|---|---|
| `final_status` | `no complaint in download`, `artifact` (e.g. an unlinked cover sheet), `not in scope`, or a status containing **found** when you find a complaint (e.g. `complaint found`, `first amended complaint found in download`) |
| `complaint_file` | only if found: the file name, copied exactly from `file_names` (either copy of a duplicated file works) |
| `complaint_pages` | only if found: e.g. `12-40`, `12-` (to the end of the file), or `all` |
| `check_note` | why, e.g. `appellate brief only`, `complaint after docket report` |

Save it back as CSV with the same name (File → Download → CSV, rename, upload to the output folder).

**4c.** Transfer the complaints you found into `manifest.csv` (run once; running it again is harmless, as complaints already transferred are skipped). Any status containing "found" (but not "no complaint") counts; amended complaints keep their category (e.g. *first amended complaint*). If you named a duplicate copy, the correction is applied to the copy the notebook keeps.

In [ ]:
review_cases = pd.read_csv(OUTPUT_DIR / "cases_without_complaint_review.csv", dtype=str, keep_default_na=False)
m = pd.read_csv(OUTPUT_DIR / "manifest.csv", dtype=str, keep_default_na=False)

status = review_cases["final_status"].str.strip().str.lower()
is_found = status.str.contains("found") & ~status.str.startswith("no ")
empty = review_cases[status == ""]
if len(empty):
    print(f"⚠ {len(empty)} case(s) have no final_status yet")

applied, already, problems = 0, 0, []
for _, r in review_cases[is_found].iterrows():
    # already transferred in an earlier run? (the note starts with "final check" on the case's primary complaint)
    done = m[(m["override_note"].str.startswith("final check")) & (m["case"] == r["case"])]
    if len(done):
        already += 1
        continue
    name = r["complaint_file"].strip()
    if name and not name.lower().endswith(".pdf"):
        name += ".pdf"
    rows = m.index[(m["case_key"] == r["case_key"]) & (m["file"] == name)]
    if len(rows) and m.at[rows[0], "duplicate_of"]:                    # a duplicate copy: use the kept one
        rows = m.index[m["path"] == m.at[rows[0], "duplicate_of"]]
    if len(rows) != 1:
        problems.append(f"{r['case']}: file '{r['complaint_file']}' not found (copy the exact name from file_names)")
        continue
    s = r["final_status"].strip().lower()
    i = rows[0]
    m.at[i, "override_category"] = s.split(" found")[0] if "amended" in s else "complaint"
    m.at[i, "override_primary"] = "yes"
    m.at[i, "override_pages"] = r["complaint_pages"].strip()
    m.at[i, "override_note"] = f"final check: {r['final_status'].strip()}; {r['check_note'].strip()}".strip("; ")
    applied += 1

m.to_csv(OUTPUT_DIR / "manifest.csv", index=False, encoding="utf-8-sig")
print(f"{applied} found complaint(s) added to manifest.csv; {already} already added in an earlier run "
      f"({int(is_found.sum())} found in total)")
for p in problems:
    print("⚠", p)
print("\nFinal status of the reviewed cases:")
print(status.value_counts().to_string())

**4d.** Re-run **Step 1** and **Step 3**: the complaints you found are now classified and copied, and the cases still without a complaint are the ones you marked otherwise.

## Step 5: Validation Summary

Numbers for the methods description and README: what was processed, how each complaint was identified, and what the manual review changed. A manual override that only **confirms** the notebook's own classification (e.g. "complaint, checked by hand" on a complaint the notebook had already found) is counted as automatic, so the share of complaints found automatically is not understated; real corrections are counted separately.

In [ ]:
m = pd.read_csv(OUTPUT_DIR / "manifest.csv", dtype=str, keep_default_na=False)
m["is_primary"] = m["primary"].str.strip().str.lower() == "true"
n_cases = m["case_key"].nunique()
prim = m[m["is_primary"]].copy()

# How each primary complaint was identified. A manual override that merely confirms the notebook's own
# classification counts as automatic (confirmed); the notebook's original reasoning is kept after the override note.
prim["manual"] = prim["override_category"].str.strip() != ""
prim["final_check"] = prim["override_note"].str.startswith("final check")
auto_reason = prim["reasons"].str.replace(r"^(pages set manually \([^)]*\); )?manual override \([^)]*\); ", "",
                                          regex=True)
auto_said_complaint = (auto_reason.str.contains(r"^title: '[^']*(?:complaint|petition)", case=False, regex=True)
                       | auto_reason.str.contains("complaint found inside", regex=False))
prim["how"] = "found automatically"
prim.loc[prim["manual"] & auto_said_complaint, "how"] = "found automatically, confirmed by hand"
prim.loc[prim["manual"] & ~auto_said_complaint, "how"] = "corrected by hand"
prim.loc[prim["final_check"], "how"] = "found in the final check"

overrides = m[m["override_category"].str.strip() != ""]
o_reason = overrides["reasons"].str.replace(r"^(pages set manually \([^)]*\); )?manual override \([^)]*\); ", "",
                                            regex=True)
confirmations = (overrides["override_category"].str.contains("complaint|petition", case=False)
                 & (o_reason.str.contains(r"^title: '[^']*(?:complaint|petition)", case=False, regex=True)
                    | o_reason.str.contains("complaint found inside", regex=False)))

print(f"PDFs processed:                  {len(m)}  (duplicates set aside: {(m['duplicate_of'] != '').sum()})")
print(f"OCR'd documents:                 {(m['ocr_applied'].str.lower() == 'true').sum()}")
print(f"Cases:                           {n_cases}")
print(f"Cases with a primary complaint:  {prim['case_key'].nunique()} ({prim['case_key'].nunique() / n_cases:.0%})")
for how, n in prim["how"].value_counts().items():
    print(f"  {how + ':':38s}{n}")
print(f"  of which found inside other documents (summons, removal, bundle): "
      f"{prim['reasons'].str.contains('found inside').sum()}")
print(f"Cases without a complaint:       {n_cases - prim['case_key'].nunique()}")
print(f"Manual overrides:                {len(overrides)}  "
      f"({confirmations.sum()} confirmations, {len(overrides) - confirmations.sum()} corrections)")
print(f"Manual page-range corrections:   {(m['override_pages'].str.strip() != '').sum()}")

review_path = OUTPUT_DIR / "cases_without_complaint_review.csv"
if review_path.exists():
    rc = pd.read_csv(review_path, dtype=str, keep_default_na=False)
    print("\nFinal check of cases without a complaint:")
    print(rc["final_status"].str.strip().str.lower().value_counts().to_string())

print("\nSample of complaints corrected by hand (check that these were real corrections):")
display(prim.loc[prim["how"] == "corrected by hand", ["case", "override_note", "reasons"]].head(10))

## Step 6: Validation Test Set (Gold Set)

A **held-out test set** of randomly chosen complaints, hand-coded and compared with the pipeline's output to measure accuracy: for the sorting stage (is each selected file the case's operative, complete complaint?) and for the extraction stage (each research variable).

- **Sample:** a simple random sample of 50 primary complaints (one per case), drawn once with a fixed seed (2026) into `validation_set_50.xlsx`; the seed, the draw date and the population size are recorded in the workbook. It is **not redrawn**: the library has been re-sorted since (475 → 466 complaints), so drawing again would give a different sample.
- **Gold set:** the first 21 sampled complaints (V01–V13, V15–V22) were coded by hand; V14 was excluded (a handwritten complaint). The coded workbook is **`gold_set_final.xlsx`** in the output folder.
- **Coded variables:** *Cases*: correct document, document title, case scope, numbers of plaintiffs and defendants, primary setting and setting detail (codebook v0.3), all settings, traffic stop, pattern or practice alleged, criminal suspicion, with page references; *Plaintiffs*: name, age, gender and its basis, race as written, with page references; *Defendants*: name, type, capacity. The Instructions sheet lists the coding rules, including those added during coding (Doe groups, stadiums, protests, parked vehicles, decedents).
- **A fair test:** the complaints were coded from the PDFs before looking at the pipeline's output, and the pipeline is not adjusted based on them. Defendant lists were prepared with AI assistance (`assisted: defendants` in the notes) and are left out of the reported comparison. A second coder for inter-coder reliability is a next step.

### Step 6a: Draw the sample (done; do not re-run)

These cells drew the 50-complaint sample and created the coding workbook. They are kept to document how the sample was drawn. Running them again does nothing while `validation_set_50.xlsx` exists (the workbook is never overwritten).

In [ ]:
from datetime import date
from openpyxl import Workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation

FONT = Font(name="Arial", size=10)
BOLD = Font(name="Arial", size=10, bold=True)
HEAD = Font(name="Arial", size=10, bold=True, color="FFFFFF")
TITLE = Font(name="Arial", size=14, bold=True)
HEAD_FILL = PatternFill("solid", start_color="1F4E78")
PREFILL = PatternFill("solid", start_color="E7E6E6")      # filled in by the script: verify, don't recode
INPUT = PatternFill("solid", start_color="FFF2CC")        # to be coded
EXAMPLE = PatternFill("solid", start_color="DDEBF7")      # example row
THIN = Side(style="thin", color="BFBFBF")
BOX = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

# ---- Codebook (allowed values) --------------------------------------------------------------------
CODEBOOK = {
    "case_scope": ["encounter", "employment", "other", "not stated"],
    "setting_l1": ["T", "P", "J", "C", "R", "I", "S", "U"],
    "setting_l2": ["T1", "T2", "T3", "T4", "P1", "P2", "P3", "J1", "J2", "J3", "C1", "C2", "C3", "C4",
                   "R1", "R2", "R3", "I1", "I2", "I3", "I4", "not stated"],
    "yes_no_ns": ["yes", "no", "not stated"],
    "yes_ns": ["yes", "not stated"],
    "suspicion": ["present", "absent", "not stated", "not applicable"],
    "gender": ["Male", "Female", "other term (see notes)", "not stated"],
    "basis": ["explicit", "inferred", "not stated"],
    "defendant_type": ["Entity", "Individual", "Doe"],
    "capacity": ["individual", "official", "both", "not stated", "not applicable"],
    "correct_document": ["yes", "no: other document", "no: wrong version", "no: incomplete pages"],
}
SETTING_LABELS = [
    ("T", "Vehicle / transportation", "T1 traffic stop · T2 highway or drug interdiction · T3 checkpoint · T4 parked or stationary vehicle"),
    ("P", "Pedestrian / public space", "P1 street stop / stop-and-frisk · P2 park or public recreation · P3 public transit"),
    ("J", "Travel hub or border", "J1 airport or air travel · J2 border or immigration enforcement · J3 rail or bus terminal"),
    ("C", "Commercial / public accommodation", "C1 retail store · C2 restaurant, bar or hotel · C3 bank · C4 other business or service"),
    ("R", "Residential", "R1 own home or property · R2 housing complex or neighborhood · R3 housing access"),
    ("I", "Institutional", "I1 school or university · I2 workplace · I3 government or public facility · I4 police station, jail or custody"),
    ("S", "Systemic (no single encounter)", "—"),
    ("U", "Unclear / not stated", "—"),
]

# ---- Columns: (header, width, kind, dropdown list or None, comment) -----------------------------
# kind: "pre" = filled by the script (verify), "in" = to be coded
CASE_COLS = [
    ("sample_id", 9, "pre", None, "Validation ID (V01-V50)."),
    ("case_number", 20, "pre", None, "Read from the complaint's court header or index number; verify against the PDF."),
    ("case_name", 34, "pre", None, "First plaintiff v. first defendant ('et al.' if more), from the sorting manifest; verify."),
    ("case_name_source", 16, "pre", None, "provider citation = legal-database citation line; caption = complaint caption; caption (LLM, verified) = LLM reading checked against the text; file name (check) = cleaned download name, often truncated: check carefully; blank = enter by hand."),
    ("date_filed", 12, "pre", None, "Read from the complaint's court header (ECF 'Filed' or NYSCEF 'FILED:'); verify, and correct if wrong."),
    ("complaint_file", 40, "pre", None, "File in the sorted complaints folder."),
    ("pages", 7, "pre", None, "Number of pages in the complaint file."),
    ("coder", 9, "in", None, "Your initials."),
    ("correct_document", 18, "in", "correct_document", "Is the file the case's operative complaint, complete? (validates the sorting stage)"),
    ("document_title", 28, "in", None, "Title as printed, e.g. FIRST AMENDED COMPLAINT."),
    ("case_scope", 12, "in", "case_scope", "encounter = a plaintiff was stopped, approached, questioned, searched, detained, arrested, followed or refused service."),
    ("n_plaintiffs", 10, "in", None, "Number of named plaintiffs (code each one on the Plaintiffs sheet)."),
    ("n_defendants", 10, "in", None, "Number of defendants listed in the caption (code each one on the Defendants sheet)."),
    ("primary_setting", 11, "in", "setting_l1", "Level-1 code of the place where the profiling contact BEGAN (see Codebook). U if not established."),
    ("setting_detail", 11, "in", "setting_l2", "Level-2 sub-code, or 'not stated' if the sub-type is unclear."),
    ("all_settings", 14, "in", None, "Every Level-1 setting of a plaintiff's encounter, separated by ';' (e.g. 'T; I')."),
    ("setting_page", 9, "in", None, "Page (PDF page number) describing the primary setting."),
    ("traffic_stop", 11, "in", "yes_no_ns", "Did any plaintiff's encounter involve a vehicle stop?"),
    ("pattern_alleged", 12, "in", "yes_ns", "Does the complaint allege a policy, practice, custom, program or pattern?"),
    ("pattern_page", 9, "in", None, "Page of the pattern allegation."),
    ("criminal_suspicion", 14, "in", "suspicion", "Commercial settings (C) only: was the plaintiff treated as a criminal suspect? Otherwise 'not applicable'."),
    ("minutes_spent", 9, "in", None, "Approximate minutes to code this complaint (for planning and reporting)."),
    ("notes", 40, "in", None, "Anything unusual, uncertain, or worth discussing."),
]
PLAINTIFF_COLS = [
    ("sample_id", 9, "in", None, "Validation ID from the Cases sheet."),
    ("case_number", 20, "in", None, "Copy from the Cases sheet."),
    ("plaintiff_no", 9, "in", None, "1, 2, 3 ... in caption order."),
    ("name", 26, "in", None, "Full name as written."),
    ("age", 7, "in", None, "Only if stated (number). Never calculate from dates; 'over eighteen' is not an age."),
    ("age_page", 8, "in", None, "Page where the age is stated."),
    ("gender", 12, "in", "gender", "As stated or indicated; never inferred from the name."),
    ("gender_basis", 11, "in", "basis", "explicit = man/woman/male/female; inferred = pronouns or honorifics only."),
    ("gender_page", 8, "in", None, "Page of the evidence."),
    ("race_as_written", 26, "in", None, "Race or ethnicity in the complaint's own words; blank if not stated. Never inferred."),
    ("race_page", 8, "in", None, "Page of the evidence."),
    ("notes", 36, "in", None, ""),
]
DEFENDANT_COLS = [
    ("sample_id", 9, "in", None, "Validation ID from the Cases sheet."),
    ("case_number", 20, "in", None, "Copy from the Cases sheet."),
    ("defendant_no", 9, "in", None, "1, 2, 3 ... in caption order."),
    ("name", 34, "in", None, "Name as written in the caption."),
    ("type", 11, "in", "defendant_type", "Entity (government body, agency, company), Individual, or Doe placeholder."),
    ("capacity", 12, "in", "capacity", "Only if stated (individual / official / both); 'not applicable' for entities and Does."),
    ("notes", 36, "in", None, ""),
]


# ---- Reading case number and filing date from the complaint ---------------------------------------
FED_RE = re.compile(r"\b\d{1,2}:\d{2}-[a-z]{2,4}-\d{3,6}(?:-[A-Z]{1,4})*\b", re.I)
ECF_FILED_RE = re.compile(r"Filed:?\s+(\d{1,2}/\d{1,2}/\d{2,4})", re.I)
NY_INDEX_RE = re.compile(r"index\s+no\.?\s*:?\s*(\d{3,7}\s*/\s*\d{2,4})", re.I)
NY_FILED_RE = re.compile(r"FILED:\s+[A-Z .]+CLERK\s+(\d{1,2}/\d{1,2}/\d{4})")
STATE_RE = re.compile(r"\b(\d{2}[A-Z]{2,6}\d{4,8}|[A-Z]{3}-[A-Z]{1,2}-\d{3,6}-\d{2})\b")


def read_header(pdf_path, pages=2):
    """Case number and filing date as printed on the complaint's first pages (blank if not found)."""
    try:
        with fitz.open(pdf_path) as doc:
            text = "\n".join(doc[i].get_text() for i in range(min(pages, len(doc))))
            n_pages = len(doc)
    except Exception:
        return "", "", None
    case_no = (FED_RE.search(text) or NY_INDEX_RE.search(text) or STATE_RE.search(text))
    case_no = (case_no.group(1) if case_no and case_no.re is not FED_RE else case_no.group(0)) if case_no else ""
    filed = ECF_FILED_RE.search(text) or NY_FILED_RE.search(text)
    filed = filed.group(1) if filed else ""
    if filed:
        try:
            filed = pd.to_datetime(filed).date()
        except Exception:
            pass
    return case_no, filed, n_pages


# ---- Workbook helpers ---------------------------------------------------------------------------------
def header_row(ws, cols, row=1):
    for j, (name, width, kind, lst, comment) in enumerate(cols, start=1):
        c = ws.cell(row=row, column=j, value=name)
        c.font, c.fill, c.border = HEAD, HEAD_FILL, BOX
        c.alignment = Alignment(wrap_text=True, vertical="center")
        if comment:
            c.comment = Comment(comment, "codebook")
        ws.column_dimensions[get_column_letter(j)].width = width
    ws.freeze_panes = ws.cell(row=row + 1, column=2)
    ws.row_dimensions[row].height = 30


def style_body(ws, cols, first, last, example_row=None):
    for r in range(first, last + 1):
        for j, (name, width, kind, lst, comment) in enumerate(cols, start=1):
            c = ws.cell(row=r, column=j)
            c.font, c.border = FONT, BOX
            c.alignment = Alignment(vertical="top", wrap_text=name in ("notes", "case_name", "complaint_file"))
            c.fill = EXAMPLE if r == example_row else (PREFILL if kind == "pre" else INPUT)


def add_dropdowns(ws, cols, first, last, lists_ref):
    for j, (name, width, kind, lst, comment) in enumerate(cols, start=1):
        if lst:
            dv = DataValidation(type="list", formula1=lists_ref[lst], allow_blank=True, showErrorMessage=True,
                                errorTitle="Not a codebook value", error="Choose a value from the list (see Codebook).")
            col = get_column_letter(j)
            dv.add(f"{col}{first}:{col}{last}")
            ws.add_data_validation(dv)


# ---- Main ---------------------------------------------------------------------------------------------
def build_validation_workbook(manifest_path, complaints_dir, out_path, n=50, seed=2026,
                              plaintiff_rows=150, defendant_rows=400, overwrite=False):
    """Draw a simple random sample of n primary complaints (fixed seed) and write the coding workbook.
    An existing workbook is never overwritten unless overwrite=True, so coding already done is safe."""
    if Path(out_path).exists() and not overwrite:
        print(f"⚠ {out_path} already exists and was NOT overwritten (your coding is safe).")
        print("  To draw it again, delete or rename the file, or pass overwrite=True.")
        return None
    m = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
    if "case_name" not in m:
        print("⚠ The manifest has no case_name column: run Step 1 and Step 3 of the current notebook first.")
        return None
    prim = m[m["primary"].str.strip().str.lower() == "true"].drop_duplicates("case_key")
    population = len(prim)
    n = min(n, population)
    sample = prim.sample(n=n, random_state=seed).reset_index(drop=True)
    complaints_dir = Path(complaints_dir)

    rows = []
    for i, r in sample.iterrows():
        case_no, filed, n_pages = read_header(complaints_dir / r["new_name"])
        rows.append({"sample_id": f"V{i + 1:02d}",
                     "case_number": case_no or r["case_number"],
                     "case_name": r.get("case_name", ""),
                     "case_name_source": r.get("case_name_source", ""),
                     "date_filed": filed,
                     "complaint_file": r["new_name"],
                     "pages": n_pages if n_pages is not None else r["pages"]})

    wb = Workbook()

    # Codebook sheet (also holds the dropdown lists)
    cb = wb.active
    cb.title = "Codebook"
    cb["A1"], cb["A1"].font = "Codebook v0.3 (lean): allowed values", TITLE
    lists_ref, col = {}, 1
    for key, values in CODEBOOK.items():
        cb.cell(row=3, column=col, value=key).font = BOLD
        for k, v in enumerate(values, start=4):
            cb.cell(row=k, column=col, value=v).font = FONT
        letter = get_column_letter(col)
        lists_ref[key] = f"=Codebook!${letter}$4:${letter}${3 + len(values)}"
        cb.column_dimensions[letter].width = max(14, max(len(v) for v in values) + 2)
        col += 1
    r0 = 4 + max(len(v) for v in CODEBOOK.values()) + 2
    cb.cell(row=r0, column=1, value="Setting codes").font = BOLD
    for k, (code, label, subs) in enumerate(SETTING_LABELS, start=r0 + 1):
        cb.cell(row=k, column=1, value=code).font = BOLD
        cb.cell(row=k, column=2, value=label).font = FONT
        cb.cell(row=k, column=3, value=subs).font = FONT
    rule_row = r0 + len(SETTING_LABELS) + 2
    rules = [
        "Primary setting = where the profiling contact BEGAN; later places (e.g. a police station) go in all_settings.",
        "Outside a business: code P unless the plaintiff was a customer or the business initiated the encounter (then C).",
        "Never infer a setting from who the defendants are (a highway patrol defendant does not establish a traffic stop).",
        "Code the level you are sure of: a vehicle encounter of unclear type is T with setting_detail 'not stated'.",
        "Scope 'employment' or 'other': leave the setting variables as 'not stated' / 'not applicable'.",
    ]
    cb.cell(row=rule_row, column=1, value="Setting rules").font = BOLD
    for k, t in enumerate(rules, start=rule_row + 1):
        cb.cell(row=k, column=1, value=t).font = FONT

    # Cases sheet
    ws = wb.create_sheet("Cases", 0)
    header_row(ws, CASE_COLS)
    example = {"sample_id": "EX", "case_number": "1:23-cv-01001-ABC", "case_name": "EXAMPLE ROW: delete or ignore",
               "case_name_source": "provider citation",
               "date_filed": date(2023, 3, 1), "complaint_file": "2023cv01001_avery_v_riverton_complaint_ecf1.pdf",
               "pages": 24, "coder": "DT", "correct_document": "yes", "document_title": "COMPLAINT AND JURY DEMAND",
               "case_scope": "encounter", "n_plaintiffs": 1, "n_defendants": 4, "primary_setting": "T",
               "setting_detail": "T1", "all_settings": "T; I", "setting_page": 5, "traffic_stop": "yes",
               "pattern_alleged": "not stated", "pattern_page": "", "criminal_suspicion": "not applicable",
               "minutes_spent": 25, "notes": "Plaintiff later taken to the precinct (I4)."}
    names = [c[0] for c in CASE_COLS]
    for j, name in enumerate(names, start=1):
        ws.cell(row=2, column=j, value=example.get(name, ""))
    for i, rec in enumerate(rows, start=3):
        for j, name in enumerate(names, start=1):
            ws.cell(row=i, column=j, value=rec.get(name, ""))
    last = 2 + len(rows)
    style_body(ws, CASE_COLS, 2, last, example_row=2)
    for r in range(2, last + 1):
        ws.cell(row=r, column=names.index("date_filed") + 1).number_format = "yyyy-mm-dd"
    add_dropdowns(ws, CASE_COLS, 3, last, lists_ref)
    ws.auto_filter.ref = f"A1:{get_column_letter(len(CASE_COLS))}{last}"

    # Plaintiffs and Defendants sheets (blank rows to fill, one example row each)
    for title, cols, n_rows, ex in [
        ("Plaintiffs", PLAINTIFF_COLS, plaintiff_rows,
         {"sample_id": "EX", "case_number": "1:23-cv-01001-ABC", "plaintiff_no": 1, "name": "Jordan Avery",
          "age": 34, "age_page": 3, "gender": "Male", "gender_basis": "explicit", "gender_page": 3,
          "race_as_written": "Black", "race_page": 3, "notes": "EXAMPLE ROW"}),
        ("Defendants", DEFENDANT_COLS, defendant_rows,
         {"sample_id": "EX", "case_number": "1:23-cv-01001-ABC", "defendant_no": 1, "name": "City of Riverton",
          "type": "Entity", "capacity": "not applicable", "notes": "EXAMPLE ROW"}),
    ]:
        sh = wb.create_sheet(title, 1 if title == "Plaintiffs" else 2)
        header_row(sh, cols)
        cn = [c[0] for c in cols]
        for j, name in enumerate(cn, start=1):
            sh.cell(row=2, column=j, value=ex.get(name, ""))
        style_body(sh, cols, 2, n_rows + 2, example_row=2)
        add_dropdowns(sh, cols, 3, n_rows + 2, lists_ref)
        sid = DataValidation(type="list", formula1=f"=Cases!$A$3:$A${last}", allow_blank=True)
        sid.add(f"A3:A{n_rows + 2}")
        sh.add_data_validation(sid)

    # Instructions sheet
    ins = wb.create_sheet("Instructions", 0)
    ins.column_dimensions["A"].width = 26
    ins.column_dimensions["B"].width = 100
    ins["A1"], ins["A1"].font = "Validation set: hand coding of 50 randomly selected complaints", TITLE
    info = [
        ("Purpose", "A held-out test set: hand-coded values are compared with the pipeline's output to measure accuracy "
                    "(sorting: right document, version and pages; extraction: each research variable). Do not use these "
                    "complaints to adjust prompts or rules; otherwise they stop being a fair test."),
        ("Population", f"{population} cases with a primary complaint (manifest: {Path(manifest_path).name})"),
        ("Sample", f"{n} complaints, simple random sample, seed {seed}, drawn {date.today().isoformat()}"),
        ("Code from", "The complaint PDF only (the file in complaint_file). Do not look at the pipeline's output before coding."),
        ("Grey cells", "Filled in by the script (case number and filing date from the complaint's header, case name from its "
                       "caption): verify against the PDF and correct if wrong; note corrections in 'notes'. Where "
                       "case_name_source is 'file name' or blank, the caption was not recognized: enter the name by hand."),
        ("Yellow cells", "To be coded. Use the dropdowns where present; values must match the Codebook sheet."),
        ("Blue row", "Example row showing the expected format (sample_id EX); leave it or delete it."),
        ("Not stated", "If the complaint does not state something, choose 'not stated' (or leave age/race blank). "
                       "Never infer race, gender or age from names, places or dates."),
        ("Page numbers", "PDF page numbers (page 1 = first page of the file), not the court's printed page numbers."),
        ("Plaintiffs / Defendants", "One row per person or entity, with the sample_id and case_number copied from the Cases sheet."),
        ("Second coder", "For inter-coder reliability, a second person codes a copy of this file (at least 20-30 complaints) "
                         "independently, without seeing the first coder's values."),
        ("Order", "Code in sample_id order. Record minutes_spent per complaint."),
    ]
    for k, (a, b) in enumerate(info, start=3):
        ins.cell(row=k, column=1, value=a).font = BOLD
        c = ins.cell(row=k, column=2, value=b)
        c.font, c.alignment = FONT, Alignment(wrap_text=True, vertical="top")
    for k, (fill, label) in enumerate([(PREFILL, "verify (pre-filled)"), (INPUT, "code (input)"),
                                       (EXAMPLE, "example row")], start=len(info) + 4):
        ins.cell(row=k, column=1, value=label).fill = fill
        ins.cell(row=k, column=1).font = FONT

    for sheet in wb.worksheets:
        sheet.sheet_view.zoomScale = 100
    wb.save(out_path)
    print(f"Sampled {n} of {population} complaints (seed {seed}) -> {out_path}")
    missing_dates = sum(1 for r in rows if not r["date_filed"])
    print(f"Filing date read from the complaint header for {n - missing_dates} of {n}; "
          f"{missing_dates} to fill in by hand.")
    print("Case names by source:", pd.Series([r["case_name_source"] or "none" for r in rows]).value_counts().to_dict())
    print("Check by hand the rows whose source is 'file name (check)' or blank.")
    return pd.DataFrame(rows)

**Create the workbook** (done once; skipped while the workbook exists).

In [ ]:
# Step 6a (done): drawn once; never overwritten while validation_set_50.xlsx exists
validation_path = OUTPUT_DIR / "validation_set_50.xlsx"
if validation_path.exists():
    print("Already drawn:", validation_path, "(not redrawn)")
else:
    validation_df = build_validation_workbook(OUTPUT_DIR / "manifest.csv", OUTPUT_DIR / "complaints",
                                              validation_path, n=50, seed=2026)
    if validation_df is not None:
        display(validation_df)

### Step 6b: Prepare the Gold Set for the Validation Run (run this now)

Run this after Step 3, whenever the library has been re-sorted. It:
1. finds each gold complaint's **current** file in the sorted library (file names change when the library is re-sorted), matched by case number;
2. updates `complaint_file` in `gold_set_final.xlsx` (a backup of the first version is kept as `gold_set_final_backup.xlsx`);
3. copies the gold complaints into **`validation_complaints/`**, replacing old copies. Complaints whose selected file is not the complaint (`correct_document` = no, e.g. V04, V05) go to `validation_complaints/_not_compared/`: they count for the sorting check, but are not extracted.

Expected: 21 gold complaints, about 19 copied and 2 not compared. If a line starts with ⚠, enter that complaint's file name in the gold set by hand (from `complaints/`) and run the cell again.

In [ ]:
import shutil
import openpyxl

GOLD = OUTPUT_DIR / "gold_set_final.xlsx"                  # the hand-coded gold set (21 complaints)
VALIDATION_DIR = OUTPUT_DIR / "validation_complaints"     # input folder for the extraction run
assert GOLD.exists(), f"Gold set not found: {GOLD} (upload gold_set_final.xlsx to the output folder)"


def norm_num(s):
    """Comparable case number: '2:24-cv-01234-ABC-DE' and '2024cv01234' -> '24cv1234'; '153536/2023' -> '153536_2023'."""
    s = str(s).lower().strip()
    m_ = re.search(r"\d{1,2}:(\d{2})-[a-z]{2,4}-0*(\d+)", s) or re.search(r"^20(\d{2})[a-z]{2,4}0*(\d+)", s)
    if m_:
        return f"{m_.group(1)}cv{int(m_.group(2))}"
    m_ = re.search(r"(?:ny)?(\d{3,7})\s*[/_]\s*(\d{4})", s)
    if m_:
        return f"{m_.group(1)}_{m_.group(2)}"
    return re.sub(r"[^a-z0-9]", "", s)


# 1. the current sorted file of each gold complaint (file names change when the library is re-sorted)
man = pd.read_csv(OUTPUT_DIR / "manifest.csv", dtype=str, keep_default_na=False)
prim = man[man["primary"].str.upper() == "TRUE"].copy()
prim["num"] = prim["case_number"].map(norm_num)
gold_cases = pd.read_excel(GOLD, sheet_name="Cases", dtype=str).fillna("")
gold_cases = gold_cases[gold_cases["sample_id"].str.match(r"V\d+")]
def surname(case_name):
    """First plaintiff's surname from a case name: 'Jordan Avery v. The City of Riverton et al.' -> 'avery'."""
    first = re.split(r"\s+v\.?\s+", str(case_name), maxsplit=1)[0]
    first = re.sub(r"\bet al\.?|,.*$", "", first).strip()
    words = re.findall(r"[a-z]+", first.lower())
    return words[-1] if words else ""


current, problems = {}, []
for _, r in gold_cases.iterrows():
    sur = surname(r.get("case_name", ""))
    hit = prim[prim["num"] == norm_num(r["case_number"])]
    if len(hit) > 1 and sur:                                   # several cases with this number: use the name
        hit = hit[hit["new_name"].str.contains(sur, case=False, regex=False)]
    if len(hit) == 1 and sur and sur not in hit["new_name"].iloc[0].lower():
        # number matches, name does not (e.g. a case number copied from another row): try the name instead
        by_name = prim[prim["new_name"].str.contains(sur, case=False, regex=False)]
        hit = by_name if len(by_name) == 1 else hit
    if len(hit) == 0 and sur:
        hit = prim[prim["new_name"].str.contains(sur, case=False, regex=False)]
    if len(hit) == 1:
        current[r["sample_id"]] = hit["new_name"].iloc[0]
    elif (OUTPUT_DIR / "complaints" / r["complaint_file"]).exists():
        current[r["sample_id"]] = r["complaint_file"]
    else:
        problems.append(f"{r['sample_id']} ({r['case_number']}): no unique match; enter its complaint_file by hand")

# 2. update complaint_file in the gold set (a backup of the first version is kept)
backup = GOLD.with_name(GOLD.stem + "_backup.xlsx")
if not backup.exists():
    shutil.copy2(GOLD, backup)
wb = openpyxl.load_workbook(GOLD)
ws = wb["Cases"]
col = {c.value: c.column for c in ws[1]}
changed = 0
for row in range(2, ws.max_row + 1):
    sid = ws.cell(row=row, column=col["sample_id"]).value
    if sid in current and ws.cell(row=row, column=col["complaint_file"]).value != current[sid]:
        ws.cell(row=row, column=col["complaint_file"]).value = current[sid]
        changed += 1
wb.save(GOLD)
print(f"Gold set: {len(gold_cases)} complaints; {changed} file name(s) updated (backup: {backup.name})")
for p in problems:
    print("⚠", p)

# 3. copy the gold complaints into the validation folder (old PDFs there are replaced). Complaints whose
#    selected file is not the complaint (correct_document = no) go to _not_compared/: they count for the
#    sorting check but are not extracted.
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
skip_dir = VALIDATION_DIR / "_not_compared"
for folder in (VALIDATION_DIR, skip_dir):
    if folder.exists():
        for f in folder.glob("*.pdf"):
            f.unlink()
rows = []
for _, r in gold_cases.iterrows():
    name = current.get(r["sample_id"])
    src = OUTPUT_DIR / "complaints" / name if name else None
    wrong = str(r.get("correct_document", "")).strip().lower().startswith("no")
    if src is None or not src.exists():
        rows.append((r["sample_id"], name, "not found"))
        continue
    target = skip_dir if wrong else VALIDATION_DIR
    target.mkdir(exist_ok=True)
    shutil.copy2(src, target / name)
    rows.append((r["sample_id"], name, "not compared (correct_document = no)" if wrong else "copied"))
status = pd.DataFrame(rows, columns=["sample_id", "complaint_file", "status"])
print(status["status"].value_counts().to_string())
display(status)

## Next Steps

**1. Extraction on the gold complaints**: `02_Complaint_RAG_Extraction_validation.ipynb`
- Runtime → Change runtime type → **T4 GPU**; have `config.json` (OpenAI key) ready.
- Settings: `COMPLAINT_FOLDER = "/content/drive/MyDrive/COMPLAINTS_SORTED/validation_complaints"`, `MAX_PAGES_INDEXED = 60`, and the model you will use for the whole project (keep it fixed from now on).
- Run the install cell → Runtime → Restart session → click the next code cell → **Run cell and below**. The parallel batch at the end writes `validation_complaints/library_analysis.xlsx`.

**2. Comparison**: `03_Validation.ipynb`
- Step 1 (check the gold set) → skip Step 2 (done by Step 6b here) → Steps 3–5 (compare, sensitivity check) → Step 6 (classify each disagreement; correct gold-set errors, documented) → re-run Step 4 → Step 7 (methods paragraph).

**3. The full library**: the extraction notebook with `COMPLAINT_FOLDER = "/content/drive/MyDrive/COMPLAINTS_SORTED/complaints"` (466 complaints), in batches, with the same settings as the validation run.